# End-to-End Computational Workflow  
## Digital Twin-Based Risk-Aware Routing untuk Distribusi Kosmetik Perkotaan

Notebook ini menyusun alur komputasi secara sistematis dari data Excel sumber sampai menghasilkan tabel, gambar, ringkasan hasil eksperimen, dan paket output akhir.

**Input utama**
- `baseline_dataset_prepared_coordinate_fixed.xlsx`

**Alur proses**
1. Load dan audit data dari Excel.
2. Standardisasi tanggal, waktu, koordinat, berat, dan volume.
3. Agregasi order menjadi customer-day stop.
4. Analisis dataset, time windows, berat-volume, dan densitas.
5. Pembentukan profil instance harian.
6. Pembentukan matriks jarak dan waktu tempuh berbasis koordinat.
7. Evaluasi kuantil waktu tempuh p50, p85, dan p95.
8. Penyusunan desain skenario S0 sampai S5.
9. Rekap hasil eksperimen final S0 sampai S4.
10. Analisis improvement, trade-off, robustness awal, dan diagnostic feedback.
11. Ekspor seluruh tabel, gambar, audit, dan ringkasan ke ZIP.

Notebook ini dibuat agar proses penelitian dapat ditelusuri dari input data sampai output akhir secara rapi dan dapat dijalankan ulang.

In [ ]:
# ============================================================
# CELL 1 — SETUP ENVIRONMENT
# ============================================================

import os, re, math, json, zipfile, shutil, textwrap, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch, Polygon

warnings.filterwarnings("ignore")

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

OUTPUT_DIR = Path("/content/dt_routing_outputs")
TABLE_DIR = OUTPUT_DIR / "01_tables_csv"
TABLE_IMG_DIR = OUTPUT_DIR / "02_tables_png"
FIG_DIR = OUTPUT_DIR / "03_figures_png"
AUDIT_DIR = OUTPUT_DIR / "04_audit_outputs"
MANIFEST_DIR = OUTPUT_DIR / "99_manifest"

for d in [TABLE_DIR, TABLE_IMG_DIR, FIG_DIR, AUDIT_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120

print("Output folder:", OUTPUT_DIR)
print("Run date:", datetime.now())

In [ ]:
# ============================================================
# CELL 2 — INPUT FILE DAN PARAMETER UTAMA
# ============================================================

EXCEL_NAME = "baseline_dataset_prepared_coordinate_fixed.xlsx"
EXCEL_PATH = Path("/content") / EXCEL_NAME

# Saved matrix package dari notebook OSRM + TomTom Matrix Builder.
# Notebook ini dapat membaca nama:
# - osrm_tomtom_saved_matrices.zip
# - osrm_tomtom_saved_matrices (1).zip
# - atau variasi lain yang diawali osrm_tomtom_saved_matrices dan berakhiran .zip
MATRIX_ZIP_NAME = "osrm_tomtom_saved_matrices.zip"
MATRIX_ZIP_PATH = Path("/content") / MATRIX_ZIP_NAME
MATRIX_ROOT = Path("/content/osrm_tomtom_saved_matrices")

# File utama di dalam ZIP matrix.
COMBINED_DISTANCE_NAME = "combined_osrm_distance_matrix_km_long.csv"
COMBINED_DURATION_NAME = "combined_osrm_duration_matrix_min_long.csv"
COMBINED_TT_P50_NAME = "combined_travel_time_matrix_p50_long.csv"
COMBINED_TT_P85_NAME = "combined_travel_time_matrix_p85_long.csv"
COMBINED_TT_P95_NAME = "combined_travel_time_matrix_p95_long.csv"
TOMTOM_MULTIPLIER_NAME = "tomtom_traffic_multiplier.csv"
MATRIX_SOURCE_AUDIT_NAME = "matrix_source_audit.csv"

# Mode konsistensi:
# True = notebook membaca saved OSRM + TomTom matrix package.
REQUIRE_SAVED_MATRIX_PACKAGE = True

# Kapasitas kendaraan.
DEFAULT_CAPACITY_WEIGHT_KG = 10000.0
DEFAULT_CAPACITY_VOLUME_CM3 = 3992800.0

def autodetect_matrix_zip():
    candidates = sorted(Path("/content").glob("osrm_tomtom_saved_matrices*.zip"))
    if MATRIX_ZIP_PATH.exists():
        return MATRIX_ZIP_PATH
    if candidates:
        # Pilih file terbesar jika ada beberapa versi upload.
        candidates = sorted(candidates, key=lambda p: p.stat().st_size, reverse=True)
        return candidates[0]
    return MATRIX_ZIP_PATH

required_excel_missing = not EXCEL_PATH.exists()
matrix_zip_detected = autodetect_matrix_zip()
required_matrix_missing = not matrix_zip_detected.exists()

if IN_COLAB and (required_excel_missing or required_matrix_missing):
    print("File yang belum ditemukan:")
    if required_excel_missing:
        print("-", EXCEL_NAME)
    if required_matrix_missing:
        print("-", "osrm_tomtom_saved_matrices*.zip")
    print("\nSilakan upload file Excel sumber dan ZIP saved matrix.")
    uploaded = files.upload()

matrix_zip_detected = autodetect_matrix_zip()
MATRIX_ZIP_PATH = matrix_zip_detected

print("Excel exists:", EXCEL_PATH.exists(), EXCEL_PATH)
print("Saved matrix ZIP detected:", MATRIX_ZIP_PATH.exists(), MATRIX_ZIP_PATH)
print("REQUIRE_SAVED_MATRIX_PACKAGE:", REQUIRE_SAVED_MATRIX_PACKAGE)

In [ ]:
# ============================================================
# CELL 3 — HELPER FUNCTIONS
# ============================================================

def slugify(text, max_len=90):
    text = str(text).replace("–", "-").replace("—", "-")
    text = re.sub(r"[^\w\s\-.]+", "", text, flags=re.UNICODE)
    text = re.sub(r"\s+", "_", text.strip())
    return text[:max_len].strip("_")

def save_df(df, title, prefix="Tabel"):
    safe = slugify(title)
    csv_path = TABLE_DIR / f"{prefix}_{safe}.csv"
    png_path = TABLE_IMG_DIR / f"{prefix}_{safe}.png"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    save_table_png(df, title, png_path)
    return csv_path, png_path

def save_table_png(df, title, path):
    nrows, ncols = df.shape
    fig_w = min(max(8, ncols * 2.2), 22)
    fig_h = min(max(2.6, (nrows + 1) * 0.42), 22)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")
    ax.set_title(title, fontsize=12, fontweight="bold", pad=12)
    tbl = ax.table(
        cellText=df.astype(str).values,
        colLabels=[str(c) for c in df.columns],
        cellLoc="left",
        colLoc="center",
        loc="center"
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1, 1.25)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(0.45)
        if r == 0:
            cell.set_text_props(weight="bold")
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

def save_fig(fig, title):
    path = FIG_DIR / f"{slugify(title)}.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return path

def add_box(ax, x, y, w, h, title, body="", title_size=10, body_size=8.5):
    ax.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=1.15))
    ax.text(x+w/2, y+h-0.12, title, ha="center", va="top", fontsize=title_size, fontweight="bold")
    if body:
        ax.text(x+w/2, y+h/2-0.05, body, ha="center", va="center", fontsize=body_size, linespacing=1.18)

def add_arrow(ax, x1, y1, x2, y2, ms=14):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="->", mutation_scale=ms, linewidth=1.15))

def save_flow(title, boxes, filename=None, figsize=(13,5)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 6)
    ax.axis("off")
    ax.text(6, 5.65, title, ha="center", fontsize=13, fontweight="bold")
    ax.plot([0.6, 11.4], [5.30, 5.30], linewidth=0.8)
    n = len(boxes)
    gap = 0.25
    w = (10.8 - gap*(n-1)) / n
    x0 = 0.6
    y = 3.0
    for i, (head, body) in enumerate(boxes):
        x = x0 + i*(w+gap)
        add_box(ax, x, y, w, 1.35, head, body, title_size=8.8, body_size=7.4)
        if i < n-1:
            add_arrow(ax, x+w, y+0.67, x+w+gap*0.85, y+0.67)
    path = FIG_DIR / (filename if filename else f"{slugify(title)}.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return path

def pct_change(new, old):
    return (new - old) / old * 100 if old != 0 else np.nan

def improvement_lower_is_better(new, old):
    return (old - new) / old * 100 if old != 0 else np.nan

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0088
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def time_to_min(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    try:
        parts = s.split(":")
        return int(parts[0])*60 + int(parts[1])
    except Exception:
        try:
            t = pd.to_datetime(s).time()
            return t.hour*60 + t.minute
        except Exception:
            return np.nan

print("Helper functions ready.")

In [ ]:
# ============================================================
# CELL 4 — LOAD EXCEL SOURCE
# ============================================================

if not EXCEL_PATH.exists():
    raise FileNotFoundError(f"File Excel belum ditemukan: {EXCEL_PATH}")

xl = pd.ExcelFile(EXCEL_PATH)
print("Sheet tersedia:", xl.sheet_names)

stops_raw = pd.read_excel(EXCEL_PATH, sheet_name="Stops")
depot = pd.read_excel(EXCEL_PATH, sheet_name="Depot")
params = pd.read_excel(EXCEL_PATH, sheet_name="Baseline_Params")

try:
    qa_log = pd.read_excel(EXCEL_PATH, sheet_name="Coordinate_QA_Log")
except Exception:
    qa_log = pd.DataFrame()

display(stops_raw.head())
display(depot)
display(params)
print("Jumlah baris Stops:", len(stops_raw))

In [ ]:
# ============================================================
# CELL 5 — STANDARDISASI, KONVERSI SATUAN, DAN AUDIT DATA
# ============================================================

stops = stops_raw.copy()

stops["first_delivery_dt"] = pd.to_datetime(stops["first_delivery_dt"], errors="coerce")
stops["delivery_date"] = stops["first_delivery_dt"].dt.date

stops["tw_start_str"] = stops["tw_start"].astype(str)
stops["tw_end_str"] = stops["tw_end"].astype(str)
stops["tw_start_min"] = stops["tw_start_str"].apply(time_to_min)
stops["tw_end_min"] = stops["tw_end_str"].apply(time_to_min)
stops["tw_duration_hour"] = (stops["tw_end_min"] - stops["tw_start_min"]) / 60

for col in ["longitude", "latitude", "total_weight", "total_volume", "service_time_min_baseline"]:
    stops[col] = pd.to_numeric(stops[col], errors="coerce")

stops["total_volume_m3"] = stops["total_volume"] / 1_000_000
stops["density_kg_m3"] = stops["total_weight"] / stops["total_volume_m3"].replace(0, np.nan)

quality_checks = pd.DataFrame({
    "Komponen": [
        "Rows Stops", "Missing customer_code", "Missing longitude", "Missing latitude",
        "Missing total_weight", "Missing total_volume", "Missing delivery_date",
        "Missing tw_start_min", "Missing tw_end_min"
    ],
    "Jumlah": [
        len(stops), stops["customer_code"].isna().sum(), stops["longitude"].isna().sum(),
        stops["latitude"].isna().sum(), stops["total_weight"].isna().sum(),
        stops["total_volume"].isna().sum(), stops["delivery_date"].isna().sum(),
        stops["tw_start_min"].isna().sum(), stops["tw_end_min"].isna().sum()
    ]
})

save_df(quality_checks, "Audit Kualitas Data Awal dari Excel", "audit")
display(quality_checks)

In [ ]:
# ============================================================
# CELL 6 — AGREGASI ORDER MENJADI CUSTOMER-DAY STOP
# ============================================================

agg = stops.groupby(["delivery_date", "customer_code"], as_index=False).agg(
    customer_name=("customer_name", "first"),
    longitude=("longitude", "first"),
    latitude=("latitude", "first"),
    order_count=("order_number", "count"),
    total_weight=("total_weight", "sum"),
    total_volume=("total_volume", "sum"),
    total_volume_m3=("total_volume_m3", "sum"),
    quantity_order=("quantity_order", "sum"),
    tw_start_min=("tw_start_min", "min"),
    tw_end_min=("tw_end_min", "min"),
    service_time_min_baseline=("service_time_min_baseline", "max")
)

agg["tw_duration_hour"] = (agg["tw_end_min"] - agg["tw_start_min"]) / 60
agg["density_kg_m3"] = agg["total_weight"] / agg["total_volume_m3"].replace(0, np.nan)

agg.to_csv(AUDIT_DIR / "customer_day_stops_preprocessed.csv", index=False, encoding="utf-8-sig")

print("Order awal:", len(stops))
print("Customer-day stop:", len(agg))
print("Reduksi:", len(stops) - len(agg), "atau", round((len(stops)-len(agg))/len(stops)*100, 2), "%")
display(agg.head())

In [ ]:
# ============================================================
# CELL 7 — TABEL DATASET DAN PRA-PEMROSESAN
# ============================================================

n_orders = len(stops)
n_customers = stops["customer_code"].nunique()
n_coords = stops[["longitude", "latitude"]].drop_duplicates().shape[0]
n_stops = len(agg)
n_days = agg["delivery_date"].nunique()
period = f"{pd.to_datetime(stops['delivery_date']).min().strftime('%d %B %Y')}–{pd.to_datetime(stops['delivery_date']).max().strftime('%d %B %Y')}"

daily_order = stops.groupby("delivery_date").size()
daily_stop = agg.groupby("delivery_date").size()
daily_weight = agg.groupby("delivery_date")["total_weight"].sum()
daily_volume = agg.groupby("delivery_date")["total_volume_m3"].sum()

total_weight = stops["total_weight"].sum()
total_volume_cm3 = stops["total_volume"].sum()
total_volume_m3 = total_volume_cm3 / 1_000_000
agg_density = total_weight / total_volume_m3

t41 = pd.DataFrame({
    "Komponen Dataset": [
        "Jumlah order pengiriman", "Jumlah pelanggan unik", "Jumlah lokasi koordinat unik",
        "Jumlah customer-day stop setelah agregasi", "Jumlah hari pengiriman", "Periode data",
        "Rata-rata order per hari", "Median order per hari", "Minimum order per hari", "Maksimum order per hari",
        "Rata-rata customer-day stop per hari", "Median customer-day stop per hari", "Minimum customer-day stop per hari", "Maksimum customer-day stop per hari",
        "Total berat pengiriman", "Total volume pengiriman", "Total volume pengiriman", "Densitas agregat"
    ],
    "Nilai": [
        f"{n_orders:,} order".replace(",", "."), f"{n_customers:,} pelanggan".replace(",", "."),
        f"{n_coords:,} lokasi".replace(",", "."), f"{n_stops:,} stop".replace(",", "."),
        f"{n_days} hari", period,
        f"{daily_order.mean():.2f}".replace(".", ",") + " order", f"{daily_order.median():.0f} order",
        f"{daily_order.min():.0f} order", f"{daily_order.max():.0f} order",
        f"{daily_stop.mean():.2f}".replace(".", ",") + " stop", f"{daily_stop.median():.0f} stop",
        f"{daily_stop.min():.0f} stop", f"{daily_stop.max():.0f} stop",
        f"{total_weight:,.2f} kg".replace(",", "X").replace(".", ",").replace("X", "."),
        f"{total_volume_cm3:,.2f} cm³".replace(",", "X").replace(".", ",").replace("X", "."),
        f"{total_volume_m3:,.2f} m³".replace(",", "X").replace(".", ",").replace("X", "."),
        f"{agg_density:,.2f} kg/m³".replace(",", "X").replace(".", ",").replace("X", ".")
    ]
})
save_df(t41, "Tabel 4.1 Ringkasan Dataset Distribusi Kosmetik", "Tabel_4_1")

t42 = pd.DataFrame({
    "Komponen": ["Order per hari", "Customer-day stop per hari", "Total berat harian", "Total volume harian"],
    "Rata-rata": [daily_order.mean(), daily_stop.mean(), daily_weight.mean(), daily_volume.mean()],
    "Median": [daily_order.median(), daily_stop.median(), daily_weight.median(), daily_volume.median()],
    "Minimum": [daily_order.min(), daily_stop.min(), daily_weight.min(), daily_volume.min()],
    "Maksimum": [daily_order.max(), daily_stop.max(), daily_weight.max(), daily_volume.max()],
})
save_df(t42, "Tabel 4.2 Statistik Permintaan Harian", "Tabel_4_2")

t43 = pd.DataFrame({
    "Level Analisis": ["Order-level", "Customer-day stop-level"],
    "Korelasi Berat-Volume": [
        stops[["total_weight","total_volume"]].corr().iloc[0,1],
        agg[["total_weight","total_volume"]].corr().iloc[0,1],
    ]
})
save_df(t43, "Tabel 4.3 Analisis Hubungan Berat dan Volume", "Tabel_4_3")

stops["density_category"] = np.where(stops["density_kg_m3"] < agg_density, "Bulky / volume-dominant", "Heavy / weight-dominant")
density_prop = stops["density_category"].value_counts(normalize=True).mul(100).reindex(["Bulky / volume-dominant", "Heavy / weight-dominant"])

t44 = pd.DataFrame({
    "Kategori Order": density_prop.index,
    "Kriteria": [f"Densitas < {agg_density:.2f} kg/m³", f"Densitas ≥ {agg_density:.2f} kg/m³"],
    "Proporsi": [f"{v:.2f}%" for v in density_prop.values]
})
save_df(t44, "Tabel 4.4 Kategorisasi Densitas Order", "Tabel_4_4")

tw_counts = agg["tw_duration_hour"].value_counts().sort_index()
t45 = pd.DataFrame({
    "Durasi Time Windows": [f"{int(k)} jam" if float(k).is_integer() else f"{k:.1f} jam" for k in tw_counts.index],
    "Jumlah Customer-Day Stop": tw_counts.values,
    "Proporsi": [f"{v/len(agg)*100:.2f}%" for v in tw_counts.values]
})
save_df(t45, "Tabel 4.5 Distribusi Durasi Time Windows", "Tabel_4_5")

n_multi = (agg["order_count"] > 1).sum()
max_order_stop = agg["order_count"].max()

corrected_coords = 291
if not qa_log.empty:
    try:
        hit = qa_log.iloc[:,0].astype(str).str.contains("Stops rows corrected", case=False, na=False)
        if hit.any():
            corrected_coords = int(qa_log.loc[hit].iloc[0,1])
    except Exception:
        pass

t46 = pd.DataFrame({
    "Tahapan Pra-Pemrosesan": [
        "Jumlah order awal", "Customer-day stop setelah agregasi", "Reduksi kunjungan potensial",
        "Persentase reduksi", "Customer-day stop dengan lebih dari satu order",
        "Maksimum order dalam satu customer-day stop", "Jumlah koordinat divalidasi/diperbaiki",
        "Jumlah hari pengiriman"
    ],
    "Hasil": [
        f"{n_orders:,} order".replace(",", "."), f"{n_stops:,} stop".replace(",", "."),
        f"{n_orders-n_stops:,} stop".replace(",", "."), f"{(n_orders-n_stops)/n_orders*100:.2f}%".replace(".", ","),
        f"{n_multi:,} stop".replace(",", "."), f"{max_order_stop} order",
        f"{corrected_coords} koordinat", f"{n_days} hari"
    ],
    "Implikasi terhadap Model": [
        "Menjadi data dasar aktivitas distribusi",
        "Menjadi unit kunjungan dalam model routing",
        "Mengurangi duplikasi kunjungan pelanggan pada tanggal yang sama",
        "Meningkatkan efisiensi pembentukan instance",
        "Menunjukkan pentingnya agregasi order pada pelanggan dan tanggal yang sama",
        "Beberapa order dapat direpresentasikan sebagai satu kunjungan",
        "Menjaga validitas matriks jarak dan waktu tempuh berbasis road-network",
        "Menjadi instance eksperimen harian"
    ]
})
save_df(t46, "Tabel 4.6 Ringkasan Hasil Pra-Pemrosesan Data", "Tabel_4_6")

order_dist = agg["order_count"].value_counts().sort_index()
t47 = pd.DataFrame({
    "Jumlah Order dalam Satu Customer-Day Stop": order_dist.index,
    "Jumlah Customer-Day Stop": order_dist.values,
    "Proporsi": [f"{v/len(agg)*100:.2f}%" for v in order_dist.values]
})
save_df(t47, "Tabel 4.7 Distribusi Order dalam Customer-Day Stop", "Tabel_4_7")

t48 = pd.DataFrame({
    "Komponen Data": ["Data order ganda pada pelanggan dan tanggal yang sama", "Koordinat pelanggan", "Berat dan volume", "Time windows", "Tanggal pengiriman"],
    "Risiko jika Tidak Diproses": ["Pelanggan dapat dikunjungi lebih dari satu kali", "Jarak dan waktu tempuh dapat salah", "Kapasitas kendaraan dapat salah dihitung", "Arrival time dan lateness tidak dapat dihitung tepat", "Instance harian tidak terbentuk konsisten"],
    "Dampak terhadap Model": ["Rute menjadi tidak efisien dan tidak realistis", "Matriks road-network menjadi tidak valid", "Dual-capacity feasibility checking tidak akurat", "Evaluasi reliabilitas menjadi lemah", "Perbandingan skenario menjadi tidak adil"]
})
save_df(t48, "Tabel 4.8 Kualitas Data dan Dampaknya terhadap Komponen Model", "Tabel_4_8")

display(t41)
print("Tabel dataset dan preprocessing selesai.")

In [ ]:
# ============================================================
# CELL 8 — GAMBAR DATASET DAN PRA-PEMROSESAN
# ============================================================

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(stops["total_weight"], stops["total_volume_m3"], s=12, alpha=0.45)
ax.set_title("Gambar 4.1 Scatter Plot Berat dan Volume per Order", fontweight="bold")
ax.set_xlabel("Berat (kg)")
ax.set_ylabel("Volume (m³)")
ax.grid(alpha=0.25)
save_fig(fig, "Gambar 4.1 Scatter Plot Berat dan Volume per Order")

fig, ax = plt.subplots(figsize=(7,5))
ax.bar(density_prop.index, density_prop.values)
ax.set_title("Gambar 4.2 Kategorisasi Densitas Order Bulky dan Heavy", fontweight="bold")
ax.set_ylabel("Proporsi (%)")
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Gambar 4.2 Kategorisasi Densitas Order Bulky dan Heavy")

daily = agg.groupby("delivery_date").agg(
    stop_count=("customer_code","count"),
    total_weight=("total_weight","sum"),
    total_volume_m3=("total_volume_m3","sum")
).reset_index()
daily["idx_weight"] = daily["total_weight"] / daily["total_weight"].mean() * 100
daily["idx_volume"] = daily["total_volume_m3"] / daily["total_volume_m3"].mean() * 100

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(pd.to_datetime(daily["delivery_date"]), daily["idx_weight"], marker="o", linewidth=1, label="Indeks Berat")
ax.plot(pd.to_datetime(daily["delivery_date"]), daily["idx_volume"], marker="o", linewidth=1, label="Indeks Volume")
ax.axhline(100, linestyle="--", linewidth=1)
ax.set_title("Gambar 4.3 Tren Harian Permintaan Berat dan Volume", fontweight="bold")
ax.set_ylabel("Indeks (rata-rata = 100)")
ax.legend()
ax.grid(alpha=0.25)
fig.autofmt_xdate()
save_fig(fig, "Gambar 4.3 Tren Harian Permintaan Berat dan Volume")

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(pd.to_datetime(daily["delivery_date"]), daily["stop_count"], marker="o", linewidth=1)
ax.set_title("Gambar 4.4 Fluktuasi Jumlah Customer-Day Stop Harian", fontweight="bold")
ax.set_ylabel("Jumlah Stop")
ax.grid(alpha=0.25)
fig.autofmt_xdate()
save_fig(fig, "Gambar 4.4 Fluktuasi Jumlah Customer-Day Stop Harian")

tw_plot = agg["tw_duration_hour"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8,5))
ax.bar([str(int(x))+" jam" for x in tw_plot.index], tw_plot.values / len(agg) * 100)
ax.set_title("Gambar 4.5 Distribusi Durasi Time Windows", fontweight="bold")
ax.set_xlabel("Durasi Time Windows")
ax.set_ylabel("Proporsi (%)")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Gambar 4.5 Distribusi Durasi Time Windows")

save_flow(
    "Gambar 4.6 Ilustrasi Agregasi Order menjadi Customer-Day Stop",
    [
        ("Data Order Mentah", "4.707 order\nbeberapa pelanggan berulang"),
        ("Group by", "tanggal pengiriman\n+ customer_code"),
        ("Agregasi", "berat & volume dijumlahkan\nTW dipertahankan"),
        ("Customer-Day Stop", "3.927 stop\nunit keputusan routing")
    ],
    "Gambar_4_6_Ilustrasi_Agregasi_Order_menjadi_Customer_Day_Stop.png",
    figsize=(12,5)
)

print("Gambar dataset selesai.")

In [ ]:
# ============================================================
# CELL 9 — PROFIL INSTANCE HARIAN DAN ROBUSTNESS AWAL
# ============================================================

instance_daily = agg.groupby("delivery_date").agg(
    stop_count=("customer_code","count"),
    total_weight=("total_weight","sum"),
    total_volume_m3=("total_volume_m3","sum"),
    tw_le_4_count=("tw_duration_hour", lambda x: (x <= 4).sum())
).reset_index()

instance_daily["tw_le_4_pct"] = instance_daily["tw_le_4_count"] / instance_daily["stop_count"] * 100
instance_daily = instance_daily.sort_values("stop_count").reset_index(drop=True)
instance_daily["tertile_rank"] = np.arange(len(instance_daily))

n = len(instance_daily)
instance_daily["Kategori"] = pd.cut(
    instance_daily["tertile_rank"],
    bins=[-1, n/3-1, 2*n/3-1, n],
    labels=["Kecil", "Sedang", "Besar"]
)

profile = instance_daily.groupby("Kategori", observed=False).agg(
    jumlah_instance=("delivery_date","count"),
    min_stop=("stop_count","min"),
    max_stop=("stop_count","max"),
    rata_stop=("stop_count","mean"),
    rata_berat=("total_weight","mean"),
    rata_volume=("total_volume_m3","mean"),
    rata_tw_le_4=("tw_le_4_pct","mean")
).reset_index()

t415 = pd.DataFrame({
    "Kategori Instance": profile["Kategori"],
    "Jumlah Instance": profile["jumlah_instance"],
    "Rentang Stop": profile["min_stop"].astype(int).astype(str) + "–" + profile["max_stop"].astype(int).astype(str),
    "Rata-rata Stop": profile["rata_stop"].round(2),
    "Rata-rata Berat": profile["rata_berat"].round(2),
    "Rata-rata Volume": profile["rata_volume"].round(2),
    "Rata-rata TW ≤ 4 Jam": profile["rata_tw_le_4"].round(2)
})

save_df(t415, "Tabel 4.15 Kategori Instance Pengujian", "Tabel_4_15")
save_df(t415, "Tabel 4.29 Profil Kelompok Instance Berdasarkan Ukuran Stop", "Tabel_4_29")

t430 = pd.DataFrame({
    "Kelompok Instance": ["Kecil", "Sedang", "Besar"],
    "Sumber Kompleksitas Utama": [
        "Proporsi TW ≤ 4 jam relatif tinggi",
        "Jumlah stop dan muatan berada pada tingkat operasional tipikal",
        "Jumlah stop tinggi, berat dan volume tinggi, risiko knock-on effect meningkat"
    ],
    "Implikasi terhadap Model": [
        "Model harus menjaga urutan kunjungan agar pelanggan dengan time windows sempit tidak terlambat",
        "Model perlu menjaga keseimbangan antara efisiensi rute dan reliabilitas",
        "Model harus mampu menangani konflik kapasitas dan akumulasi keterlambatan"
    ],
    "Relevansi terhadap S4": [
        "Risk-aware insertion membantu menjaga sensitivitas terhadap time windows",
        "QRG-ALNS-ILS menguji stabilitas performa pada kondisi harian umum",
        "Lateness-priority removal, dual-capacity checking, dan local search menjadi sangat penting"
    ]
})
save_df(t430, "Tabel 4.30 Interpretasi Robustness Awal Berdasarkan Kelompok Instance", "Tabel_4_30")

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(profile["Kategori"].astype(str), profile["rata_stop"])
ax.set_title("Gambar 4.11 Profil Ukuran Instance Pengujian", fontweight="bold")
ax.set_ylabel("Rata-rata Stop")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Gambar 4.11 Profil Ukuran Instance Pengujian")

fig, ax = plt.subplots(figsize=(12,6.5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis("off")
ax.text(6, 6.65, "Gambar 4.27 Profil Kompleksitas Instance Kecil–Sedang–Besar", ha="center", fontsize=13, fontweight="bold")
xs = [0.7, 4.4, 8.1]
for x, (_, r) in zip(xs, profile.iterrows()):
    title = f"Instance {r['Kategori']}"
    body = (
        f"{int(r['jumlah_instance'])} instance\n"
        f"{int(r['min_stop'])}–{int(r['max_stop'])} stop\n"
        f"Rata-rata stop: {r['rata_stop']:.2f}\n"
        f"Berat: {r['rata_berat']:.2f} kg\n"
        f"Volume: {r['rata_volume']:.2f} m³\n"
        f"TW ≤4 jam: {r['rata_tw_le_4']:.2f}%"
    )
    add_box(ax, x, 3.75, 3.05, 2.15, title, body, title_size=10, body_size=8)
    focus = "Time windows sempit" if str(r['Kategori'])=="Kecil" else ("Operasional tipikal" if str(r['Kategori'])=="Sedang" else "Konflik kapasitas\n& knock-on effect")
    add_box(ax, x, 2.45, 3.05, 0.75, "Fokus", focus, title_size=9, body_size=8)
    add_arrow(ax, x+1.525, 3.75, x+1.525, 3.20)
add_arrow(ax, 3.75, 4.85, 4.35, 4.85)
add_arrow(ax, 7.45, 4.85, 8.05, 4.85)
add_box(ax, 1.1, 0.75, 9.8, 1.0, "Implikasi terhadap Robustness S4",
        "S4 perlu stabil pada seluruh skala instance; risk-guided search paling krusial pada instance besar;\ndual-capacity checking dan lateness-priority removal mendukung skalabilitas",
        title_size=10, body_size=8.5)
fig.tight_layout()
save_fig(fig, "Gambar 4.27 Profil Kompleksitas Instance Kecil-Sedang-Besar")

instance_daily.to_csv(AUDIT_DIR / "instance_daily_profile_from_excel.csv", index=False, encoding="utf-8-sig")
display(t415)

In [ ]:
# ============================================================
# CELL 10 — LOAD SAVED OSRM + TOMTOM MATRIX PACKAGE BY DATE
# ============================================================

depot_lon = float(depot.loc[0, "longitude"])
depot_lat = float(depot.loc[0, "latitude"])

# ------------------------------------------------------------
# 10.1 Extract matrix ZIP
# ------------------------------------------------------------
if not MATRIX_ZIP_PATH.exists():
    raise FileNotFoundError(
        f"File {MATRIX_ZIP_NAME} belum tersedia. "
        "Jalankan notebook OSRM_TomTom_Matrix_Builder terlebih dahulu."
    )

if MATRIX_ROOT.exists():
    shutil.rmtree(MATRIX_ROOT)
MATRIX_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(MATRIX_ZIP_PATH, "r") as z:
    z.extractall(MATRIX_ROOT)

print("Saved matrix package extracted to:", MATRIX_ROOT)

def require_matrix_file(name):
    p = MATRIX_ROOT / name
    if not p.exists():
        raise FileNotFoundError(f"File wajib tidak ditemukan di package: {name}")
    return p

distance_long_path = require_matrix_file(COMBINED_DISTANCE_NAME)
duration_long_path = require_matrix_file(COMBINED_DURATION_NAME)
tt_p50_path = require_matrix_file(COMBINED_TT_P50_NAME)
tt_p85_path = require_matrix_file(COMBINED_TT_P85_NAME)
tt_p95_path = require_matrix_file(COMBINED_TT_P95_NAME)
tomtom_path = require_matrix_file(TOMTOM_MULTIPLIER_NAME)

source_audit_path = MATRIX_ROOT / MATRIX_SOURCE_AUDIT_NAME

# ------------------------------------------------------------
# 10.2 Load combined long matrices
# ------------------------------------------------------------
distance_long = pd.read_csv(distance_long_path)
duration_long = pd.read_csv(duration_long_path)
tt_long = {
    "p50": pd.read_csv(tt_p50_path),
    "p85": pd.read_csv(tt_p85_path),
    "p95": pd.read_csv(tt_p95_path),
}
tomtom_multiplier_df = pd.read_csv(tomtom_path)

for df_ in [distance_long, duration_long] + list(tt_long.values()):
    df_["delivery_date"] = pd.to_datetime(df_["delivery_date"]).dt.strftime("%Y-%m-%d")
    df_["origin_id"] = df_["origin_id"].astype(str)
    df_["destination_id"] = df_["destination_id"].astype(str)

tomtom_multiplier = dict(zip(
    tomtom_multiplier_df["quantile"].astype(str),
    pd.to_numeric(tomtom_multiplier_df["multiplier"], errors="coerce")
))

if source_audit_path.exists():
    matrix_source_audit = pd.read_csv(source_audit_path)
else:
    matrix_source_audit = pd.DataFrame()

# ------------------------------------------------------------
# 10.3 Coverage audit
# ------------------------------------------------------------
all_instance_dates = sorted(pd.to_datetime(agg["delivery_date"]).dt.strftime("%Y-%m-%d").unique().tolist())
available_dates = sorted(distance_long["delivery_date"].unique().tolist())

coverage = pd.DataFrame({
    "delivery_date": all_instance_dates,
})
coverage["matrix_available"] = coverage["delivery_date"].isin(available_dates)
coverage["status"] = np.where(coverage["matrix_available"], "available", "missing")
coverage.to_csv(AUDIT_DIR / "saved_matrix_date_coverage_audit.csv", index=False, encoding="utf-8-sig")

coverage_summary = pd.DataFrame({
    "Komponen": [
        "Total instance dates from Excel",
        "Available dates in saved matrix package",
        "Missing dates",
        "Coverage percentage",
        "TomTom multiplier p50",
        "TomTom multiplier p85",
        "TomTom multiplier p95"
    ],
    "Nilai": [
        len(all_instance_dates),
        len(available_dates),
        len(all_instance_dates) - len(available_dates),
        round(len(available_dates) / len(all_instance_dates) * 100, 2) if len(all_instance_dates) else 0,
        tomtom_multiplier.get("p50"),
        tomtom_multiplier.get("p85"),
        tomtom_multiplier.get("p95")
    ]
})
save_df(coverage_summary, "Audit Coverage Saved OSRM TomTom Matrix Package", "audit_matrix")
display(coverage_summary)

if len(available_dates) < len(all_instance_dates):
    display(pd.DataFrame({
        "Catatan": [
            "Saved matrix package belum mencakup seluruh tanggal instance.",
            "Ini aman untuk uji awal, tetapi untuk finalisasi seluruh eksperimen, jalankan Matrix Builder dengan MAX_DATES_FOR_TEST = None.",
            "Notebook tetap dapat menampilkan audit dan contoh matrix dari tanggal yang tersedia."
        ]
    }))

# ------------------------------------------------------------
# 10.4 Matrix subset builder by date
# ------------------------------------------------------------
def pivot_long_matrix_for_date(long_df, date_str, value_col):
    date_str = pd.to_datetime(date_str).strftime("%Y-%m-%d")
    sub = long_df[long_df["delivery_date"] == date_str].copy()
    if sub.empty:
        raise KeyError(f"Tanggal {date_str} tidak tersedia pada saved matrix package.")
    mat = sub.pivot(index="origin_id", columns="destination_id", values=value_col)
    mat.index = mat.index.astype(str)
    mat.columns = mat.columns.astype(str)
    return mat

def get_submatrix_by_node_ids_from_long(long_df, date_str, node_ids, value_col, matrix_name):
    mat = pivot_long_matrix_for_date(long_df, date_str, value_col)
    node_ids = [str(x) for x in node_ids]
    missing_rows = [x for x in node_ids if x not in mat.index]
    missing_cols = [x for x in node_ids if x not in mat.columns]
    if missing_rows or missing_cols:
        detail = {
            "matrix": matrix_name,
            "date": date_str,
            "n_missing_rows": len(missing_rows),
            "n_missing_cols": len(missing_cols),
            "missing_rows_sample": missing_rows[:10],
            "missing_cols_sample": missing_cols[:10]
        }
        raise KeyError(f"Node ID tidak lengkap pada saved matrix: {detail}")
    return mat.loc[node_ids, node_ids].astype(float).values

def build_instance_matrix(day_df, quantile="p50"):
    if "delivery_date" not in day_df.columns:
        raise ValueError("day_df harus memiliki kolom delivery_date.")

    date_str = pd.to_datetime(day_df["delivery_date"].iloc[0]).strftime("%Y-%m-%d")

    nodes = pd.concat([
        pd.DataFrame({
            "node_id": ["DEPOT_1"],
            "customer_code": ["DEPOT_1"],
            "longitude": [depot_lon],
            "latitude": [depot_lat],
            "total_weight": [0.0],
            "total_volume": [0.0],
            "tw_start_min": [8*60],
            "tw_end_min": [16*60],
            "service_time_min_baseline": [0.0]
        }),
        day_df[["customer_code","longitude","latitude","total_weight","total_volume","tw_start_min","tw_end_min","service_time_min_baseline"]].assign(
            node_id=lambda x: x["customer_code"].astype(str)
        )[["node_id","customer_code","longitude","latitude","total_weight","total_volume","tw_start_min","tw_end_min","service_time_min_baseline"]]
    ], ignore_index=True)

    node_ids = nodes["node_id"].astype(str).tolist()

    dist = get_submatrix_by_node_ids_from_long(
        distance_long, date_str, node_ids, "distance_km", "combined_osrm_distance_matrix_km_long"
    )

    base_duration = get_submatrix_by_node_ids_from_long(
        duration_long, date_str, node_ids, "duration_min", "combined_osrm_duration_matrix_min_long"
    )

    tt_df = tt_long[quantile]
    tt_col = f"travel_time_min_{quantile}"
    travel = get_submatrix_by_node_ids_from_long(
        tt_df, date_str, node_ids, tt_col, f"combined_travel_time_matrix_{quantile}_long"
    )

    return nodes, dist, travel, base_duration, date_str

# ------------------------------------------------------------
# 10.5 Audit sample matrix
# ------------------------------------------------------------
if not available_dates:
    raise RuntimeError("Tidak ada tanggal tersedia dalam saved matrix package.")

sample_date = available_dates[0]
sample_df = agg[pd.to_datetime(agg["delivery_date"]).dt.strftime("%Y-%m-%d") == sample_date].reset_index(drop=True)

nodes_sample, dist_sample, travel_p50_sample, base_duration_sample, sample_date_used = build_instance_matrix(sample_df, quantile="p50")
_, _, travel_p85_sample, _, _ = build_instance_matrix(sample_df, quantile="p85")
_, _, travel_p95_sample, _, _ = build_instance_matrix(sample_df, quantile="p95")

pd.DataFrame(dist_sample, index=nodes_sample["node_id"], columns=nodes_sample["node_id"]).to_csv(
    AUDIT_DIR / "sample_saved_osrm_distance_matrix_km.csv", encoding="utf-8-sig"
)
pd.DataFrame(base_duration_sample, index=nodes_sample["node_id"], columns=nodes_sample["node_id"]).to_csv(
    AUDIT_DIR / "sample_saved_osrm_duration_matrix_min.csv", encoding="utf-8-sig"
)
pd.DataFrame(travel_p50_sample, index=nodes_sample["node_id"], columns=nodes_sample["node_id"]).to_csv(
    AUDIT_DIR / "sample_saved_travel_time_matrix_p50.csv", encoding="utf-8-sig"
)
pd.DataFrame(travel_p85_sample, index=nodes_sample["node_id"], columns=nodes_sample["node_id"]).to_csv(
    AUDIT_DIR / "sample_saved_travel_time_matrix_p85.csv", encoding="utf-8-sig"
)
pd.DataFrame(travel_p95_sample, index=nodes_sample["node_id"], columns=nodes_sample["node_id"]).to_csv(
    AUDIT_DIR / "sample_saved_travel_time_matrix_p95.csv", encoding="utf-8-sig"
)

travel_source_audit = pd.DataFrame({
    "Komponen": [
        "Matrix package",
        "Distance matrix source",
        "Base duration matrix source",
        "Traffic multiplier source",
        "Sample date",
        "Sample nodes including depot",
        "Average OSRM distance sample",
        "Average OSRM duration sample",
        "Average p50 travel time sample",
        "Average p85 travel time sample",
        "Average p95 travel time sample",
    ],
    "Nilai": [
        MATRIX_ZIP_NAME,
        COMBINED_DISTANCE_NAME,
        COMBINED_DURATION_NAME,
        TOMTOM_MULTIPLIER_NAME,
        sample_date_used,
        len(nodes_sample),
        dist_sample[dist_sample > 0].mean(),
        base_duration_sample[base_duration_sample > 0].mean(),
        travel_p50_sample[travel_p50_sample > 0].mean(),
        travel_p85_sample[travel_p85_sample > 0].mean(),
        travel_p95_sample[travel_p95_sample > 0].mean(),
    ]
})
save_df(travel_source_audit, "Audit Sumber Matriks Waktu Tempuh Saved Package", "audit_matrix")
display(travel_source_audit)

if not matrix_source_audit.empty:
    display(matrix_source_audit)

print("Saved OSRM + TomTom matrix package berhasil dibaca.")

## Visual Audit OSRM dan TomTom

Bagian ini menampilkan visual tambahan untuk memeriksa matrix OSRM dan multiplier TomTom yang digunakan. Visual yang dibuat meliputi coverage tanggal, peta sebaran node sampel, heatmap jarak dan durasi OSRM, distribusi distance-duration, scatter distance-duration, serta efek multiplier p50, p85, dan p95.

In [ ]:
# ============================================================
# CELL 10B — VISUAL AUDIT OSRM DAN TOMTOM
# ============================================================

from IPython.display import display, Markdown, Image

display(Markdown("## Visual Audit OSRM dan TomTom"))

# ------------------------------------------------------------
# 10B.1 Validity check table
# ------------------------------------------------------------
validity_rows = []

# Row count consistency by date
date_matrix_stats = []
for d in available_dates:
    n_node_from_distance = distance_long[distance_long["delivery_date"] == d]["origin_id"].nunique()
    n_row_distance = len(distance_long[distance_long["delivery_date"] == d])
    n_row_duration = len(duration_long[duration_long["delivery_date"] == d])
    expected_rows = n_node_from_distance ** 2
    date_matrix_stats.append({
        "delivery_date": d,
        "n_nodes": n_node_from_distance,
        "expected_od_pairs": expected_rows,
        "distance_rows": n_row_distance,
        "duration_rows": n_row_duration,
        "matrix_shape_valid": (n_row_distance == expected_rows) and (n_row_duration == expected_rows)
    })

date_matrix_stats_df = pd.DataFrame(date_matrix_stats)
date_matrix_stats_df.to_csv(AUDIT_DIR / "osrm_matrix_shape_validity_by_date.csv", index=False, encoding="utf-8-sig")
save_df(date_matrix_stats_df, "Audit Validitas Bentuk Matriks OSRM per Tanggal", "audit_matrix")

matrix_shape_all_valid = bool(date_matrix_stats_df["matrix_shape_valid"].all()) if len(date_matrix_stats_df) else False

validity_summary = pd.DataFrame({
    "Komponen Validasi": [
        "OSRM distance matrix available",
        "OSRM duration matrix available",
        "Travel-time p50 matrix available",
        "Travel-time p85 matrix available",
        "Travel-time p95 matrix available",
        "TomTom multiplier available",
        "Matrix shape valid for available dates",
        "Coverage of all Excel dates"
    ],
    "Status": [
        "Valid" if len(distance_long) > 0 else "Not valid",
        "Valid" if len(duration_long) > 0 else "Not valid",
        "Valid" if len(tt_long.get("p50", pd.DataFrame())) > 0 else "Not valid",
        "Valid" if len(tt_long.get("p85", pd.DataFrame())) > 0 else "Not valid",
        "Valid" if len(tt_long.get("p95", pd.DataFrame())) > 0 else "Not valid",
        "Valid" if all(q in tomtom_multiplier for q in ["p50","p85","p95"]) else "Not valid",
        "Valid" if matrix_shape_all_valid else "Check",
        "Complete" if coverage["matrix_available"].all() else "Partial"
    ],
    "Catatan": [
        "Menggunakan file combined_osrm_distance_matrix_km_long.csv",
        "Menggunakan file combined_osrm_duration_matrix_min_long.csv",
        "Dibentuk dari OSRM duration × multiplier p50",
        "Dibentuk dari OSRM duration × multiplier p85",
        "Dibentuk dari OSRM duration × multiplier p95",
        "Menggunakan file tomtom_traffic_multiplier.csv",
        "Jumlah OD pairs harus sama dengan n_nodes²",
        "Jika partial, jalankan Matrix Builder untuk seluruh tanggal"
    ]
})
save_df(validity_summary, "Audit Ringkas Validitas OSRM dan TomTom", "audit_matrix")
display(validity_summary)

# ------------------------------------------------------------
# 10B.2 Coverage visual
# ------------------------------------------------------------
coverage_plot = coverage.copy()
coverage_plot["idx"] = np.arange(len(coverage_plot))
coverage_plot["available_numeric"] = coverage_plot["matrix_available"].astype(int)

fig, ax = plt.subplots(figsize=(12, 3.8))
ax.bar(coverage_plot["idx"], coverage_plot["available_numeric"])
ax.set_title("Visual Audit 1. Coverage Saved Matrix terhadap Tanggal Instance", fontweight="bold")
ax.set_xlabel("Urutan tanggal instance")
ax.set_ylabel("Matrix tersedia")
ax.set_yticks([0, 1])
ax.set_yticklabels(["Missing", "Available"])
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Visual Audit 1 Coverage Saved Matrix terhadap Tanggal Instance")

# ------------------------------------------------------------
# 10B.3 Per-date OSRM summary
# ------------------------------------------------------------
dist_nonzero = distance_long[distance_long["distance_km"] > 0].copy()
dur_nonzero = duration_long[duration_long["duration_min"] > 0].copy()

dist_stats = dist_nonzero.groupby("delivery_date").agg(
    avg_distance_km=("distance_km", "mean"),
    median_distance_km=("distance_km", "median"),
    max_distance_km=("distance_km", "max")
).reset_index()

dur_stats = dur_nonzero.groupby("delivery_date").agg(
    avg_duration_min=("duration_min", "mean"),
    median_duration_min=("duration_min", "median"),
    max_duration_min=("duration_min", "max")
).reset_index()

osrm_date_stats = dist_stats.merge(dur_stats, on="delivery_date", how="outer")
osrm_date_stats.to_csv(AUDIT_DIR / "osrm_distance_duration_summary_by_date.csv", index=False, encoding="utf-8-sig")
save_df(osrm_date_stats, "Ringkasan Jarak dan Durasi OSRM per Tanggal", "audit_matrix")
display(osrm_date_stats)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(osrm_date_stats["delivery_date"], osrm_date_stats["avg_distance_km"])
ax.set_title("Visual Audit 2. Rata-rata OSRM Distance per Tanggal", fontweight="bold")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Rata-rata distance (km)")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Visual Audit 2 Rata-rata OSRM Distance per Tanggal")

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(osrm_date_stats["delivery_date"], osrm_date_stats["avg_duration_min"])
ax.set_title("Visual Audit 3. Rata-rata OSRM Duration per Tanggal", fontweight="bold")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Rata-rata duration (menit)")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Visual Audit 3 Rata-rata OSRM Duration per Tanggal")

# ------------------------------------------------------------
# 10B.4 Distribution and scatter
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(dist_nonzero["distance_km"].dropna(), bins=30)
ax.set_title("Visual Audit 4. Distribusi OSRM Distance OD Pairs", fontweight="bold")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Frekuensi")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Visual Audit 4 Distribusi OSRM Distance OD Pairs")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(dur_nonzero["duration_min"].dropna(), bins=30)
ax.set_title("Visual Audit 5. Distribusi OSRM Duration OD Pairs", fontweight="bold")
ax.set_xlabel("Duration (menit)")
ax.set_ylabel("Frekuensi")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Visual Audit 5 Distribusi OSRM Duration OD Pairs")

scatter_df = distance_long.merge(
    duration_long,
    on=["delivery_date", "origin_id", "destination_id"],
    how="inner"
)
scatter_df = scatter_df[(scatter_df["distance_km"] > 0) & (scatter_df["duration_min"] > 0)].copy()
scatter_df["implied_speed_kmph"] = scatter_df["distance_km"] / (scatter_df["duration_min"] / 60)

scatter_sample = scatter_df.sample(min(5000, len(scatter_df)), random_state=42) if len(scatter_df) else scatter_df

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(scatter_sample["distance_km"], scatter_sample["duration_min"], s=10, alpha=0.45)
ax.set_title("Visual Audit 6. Hubungan OSRM Distance dan Duration", fontweight="bold")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Duration (menit)")
ax.grid(alpha=0.25)
save_fig(fig, "Visual Audit 6 Hubungan OSRM Distance dan Duration")

speed_summary = scatter_df.groupby("delivery_date").agg(
    avg_implied_speed_kmph=("implied_speed_kmph", "mean"),
    median_implied_speed_kmph=("implied_speed_kmph", "median"),
    min_implied_speed_kmph=("implied_speed_kmph", "min"),
    max_implied_speed_kmph=("implied_speed_kmph", "max")
).reset_index()
speed_summary.to_csv(AUDIT_DIR / "osrm_implied_speed_summary_by_date.csv", index=False, encoding="utf-8-sig")
save_df(speed_summary, "Ringkasan Implied Speed dari OSRM Distance dan Duration", "audit_matrix")
display(speed_summary)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(speed_summary["delivery_date"], speed_summary["avg_implied_speed_kmph"])
ax.set_title("Visual Audit 7. Rata-rata Implied Speed dari OSRM", fontweight="bold")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Implied speed (km/jam)")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Visual Audit 7 Rata-rata Implied Speed dari OSRM")

# ------------------------------------------------------------
# 10B.5 Sample map-like node scatter
# ------------------------------------------------------------
sample_nodes_for_plot = nodes_sample.copy()
sample_nodes_for_plot["is_depot"] = sample_nodes_for_plot["node_id"].astype(str).eq("DEPOT_1")

fig, ax = plt.subplots(figsize=(8, 6))
cust_nodes = sample_nodes_for_plot[~sample_nodes_for_plot["is_depot"]]
dep_nodes = sample_nodes_for_plot[sample_nodes_for_plot["is_depot"]]

ax.scatter(cust_nodes["longitude"], cust_nodes["latitude"], s=30, alpha=0.7, label="Customer")
ax.scatter(dep_nodes["longitude"], dep_nodes["latitude"], s=120, marker="*", label="Depot")
ax.set_title(f"Visual Audit 8. Sebaran Node Sampel OSRM — {sample_date_used}", fontweight="bold")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(alpha=0.25)
save_fig(fig, "Visual Audit 8 Sebaran Node Sampel OSRM")

# ------------------------------------------------------------
# 10B.6 Heatmap sample distance and duration
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(dist_sample)
ax.set_title(f"Visual Audit 9. Heatmap OSRM Distance — {sample_date_used}", fontweight="bold")
ax.set_xlabel("Destination node index")
ax.set_ylabel("Origin node index")
fig.colorbar(im, ax=ax, label="Distance (km)")
save_fig(fig, "Visual Audit 9 Heatmap OSRM Distance Sample")

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(base_duration_sample)
ax.set_title(f"Visual Audit 10. Heatmap OSRM Duration — {sample_date_used}", fontweight="bold")
ax.set_xlabel("Destination node index")
ax.set_ylabel("Origin node index")
fig.colorbar(im, ax=ax, label="Duration (menit)")
save_fig(fig, "Visual Audit 10 Heatmap OSRM Duration Sample")

# ------------------------------------------------------------
# 10B.7 TomTom multiplier visuals
# ------------------------------------------------------------
tomtom_plot_df = tomtom_multiplier_df.copy()
tomtom_plot_df["quantile"] = tomtom_plot_df["quantile"].astype(str)
tomtom_plot_df["multiplier"] = pd.to_numeric(tomtom_plot_df["multiplier"], errors="coerce")

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(tomtom_plot_df["quantile"], tomtom_plot_df["multiplier"])
ax.set_title("Visual Audit 11. TomTom-Based Traffic Multiplier", fontweight="bold")
ax.set_xlabel("Quantile")
ax.set_ylabel("Multiplier")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Visual Audit 11 TomTom-Based Traffic Multiplier")

tt_avg_rows = []
for q, dfq in tt_long.items():
    col = f"travel_time_min_{q}"
    temp = dfq[dfq[col] > 0].groupby("delivery_date").agg(avg_travel_time_min=(col, "mean")).reset_index()
    temp["quantile"] = q
    tt_avg_rows.append(temp)
tt_avg = pd.concat(tt_avg_rows, ignore_index=True) if tt_avg_rows else pd.DataFrame()
tt_avg.to_csv(AUDIT_DIR / "travel_time_average_by_date_quantile.csv", index=False, encoding="utf-8-sig")
save_df(tt_avg, "Rata-rata Travel Time Berdasarkan Tanggal dan Kuantil", "audit_matrix")

fig, ax = plt.subplots(figsize=(9, 5))
for q in ["p50", "p85", "p95"]:
    sub = tt_avg[tt_avg["quantile"] == q]
    ax.plot(sub["delivery_date"], sub["avg_travel_time_min"], marker="o", label=q)
ax.set_title("Visual Audit 12. Rata-rata Travel Time p50-p85-p95 per Tanggal", fontweight="bold")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Rata-rata travel time (menit)")
ax.tick_params(axis="x", rotation=30)
ax.legend()
ax.grid(alpha=0.25)
save_fig(fig, "Visual Audit 12 Rata-rata Travel Time p50-p85-p95 per Tanggal")

display(Markdown("### Visual audit selesai. Semua gambar audit OSRM/TomTom telah disimpan ke folder figures dan akan muncul pada preview output."))

## Traffic Penalty dan Vehicle-Profile Limitation Audit

Bagian ini menambahkan visual khusus untuk menunjukkan dampak kemacetan sebagai **tambahan waktu tempuh** dari p50 ke p85 dan p95.  
Selain itu, bagian ini juga menambahkan tabel audit batasan vehicle profile agar penggunaan OSRM `driving` dan TomTom multiplier lebih transparan.

In [ ]:
# ============================================================
# CELL 10C — TRAFFIC PENALTY DAN VEHICLE-PROFILE LIMITATION AUDIT
# ============================================================

from IPython.display import display, Markdown

display(Markdown("## Traffic Penalty dan Vehicle-Profile Limitation Audit"))

# ------------------------------------------------------------
# 10C.1 Merge travel-time matrices p50, p85, p95
# ------------------------------------------------------------
tt_p50 = tt_long["p50"].copy()
tt_p85 = tt_long["p85"].copy()
tt_p95 = tt_long["p95"].copy()

tt_p50["delivery_date"] = pd.to_datetime(tt_p50["delivery_date"]).dt.strftime("%Y-%m-%d")
tt_p85["delivery_date"] = pd.to_datetime(tt_p85["delivery_date"]).dt.strftime("%Y-%m-%d")
tt_p95["delivery_date"] = pd.to_datetime(tt_p95["delivery_date"]).dt.strftime("%Y-%m-%d")

traffic_penalty = (
    tt_p50[["delivery_date", "origin_id", "destination_id", "travel_time_min_p50"]]
    .merge(tt_p85[["delivery_date", "origin_id", "destination_id", "travel_time_min_p85"]],
           on=["delivery_date", "origin_id", "destination_id"], how="inner")
    .merge(tt_p95[["delivery_date", "origin_id", "destination_id", "travel_time_min_p95"]],
           on=["delivery_date", "origin_id", "destination_id"], how="inner")
)

# Hilangkan self-loop agar statistik tidak didominasi nilai 0.
traffic_penalty = traffic_penalty[
    traffic_penalty["origin_id"].astype(str) != traffic_penalty["destination_id"].astype(str)
].copy()

traffic_penalty["penalty_p85_min"] = traffic_penalty["travel_time_min_p85"] - traffic_penalty["travel_time_min_p50"]
traffic_penalty["penalty_p95_min"] = traffic_penalty["travel_time_min_p95"] - traffic_penalty["travel_time_min_p50"]
traffic_penalty["ratio_p85_to_p50"] = traffic_penalty["travel_time_min_p85"] / traffic_penalty["travel_time_min_p50"].replace(0, np.nan)
traffic_penalty["ratio_p95_to_p50"] = traffic_penalty["travel_time_min_p95"] / traffic_penalty["travel_time_min_p50"].replace(0, np.nan)

traffic_penalty.to_csv(AUDIT_DIR / "traffic_penalty_od_pairs.csv", index=False, encoding="utf-8-sig")

display(Markdown("### Preview Traffic Penalty OD Pairs"))
display(traffic_penalty.head(20))

# ------------------------------------------------------------
# 10C.2 Summary traffic penalty overall
# ------------------------------------------------------------
traffic_penalty_summary = pd.DataFrame({
    "Metrik": [
        "Mean p50 travel time",
        "Mean p85 travel time",
        "Mean p95 travel time",
        "Mean p85 penalty over p50",
        "Mean p95 penalty over p50",
        "Median p85 penalty over p50",
        "Median p95 penalty over p50",
        "Mean ratio p85/p50",
        "Mean ratio p95/p50",
        "Max p85 penalty over p50",
        "Max p95 penalty over p50"
    ],
    "Nilai": [
        traffic_penalty["travel_time_min_p50"].mean(),
        traffic_penalty["travel_time_min_p85"].mean(),
        traffic_penalty["travel_time_min_p95"].mean(),
        traffic_penalty["penalty_p85_min"].mean(),
        traffic_penalty["penalty_p95_min"].mean(),
        traffic_penalty["penalty_p85_min"].median(),
        traffic_penalty["penalty_p95_min"].median(),
        traffic_penalty["ratio_p85_to_p50"].mean(),
        traffic_penalty["ratio_p95_to_p50"].mean(),
        traffic_penalty["penalty_p85_min"].max(),
        traffic_penalty["penalty_p95_min"].max()
    ]
})
traffic_penalty_summary["Nilai"] = traffic_penalty_summary["Nilai"].round(4)

traffic_penalty_summary.to_csv(AUDIT_DIR / "traffic_penalty_summary_overall.csv", index=False, encoding="utf-8-sig")
save_df(traffic_penalty_summary, "Traffic Penalty Summary Overall", "audit_traffic")
display(traffic_penalty_summary)

# ------------------------------------------------------------
# 10C.3 Summary traffic penalty by date
# ------------------------------------------------------------
traffic_penalty_by_date = traffic_penalty.groupby("delivery_date").agg(
    mean_p50_travel_time=("travel_time_min_p50", "mean"),
    mean_p85_travel_time=("travel_time_min_p85", "mean"),
    mean_p95_travel_time=("travel_time_min_p95", "mean"),
    mean_penalty_p85_min=("penalty_p85_min", "mean"),
    mean_penalty_p95_min=("penalty_p95_min", "mean"),
    median_penalty_p85_min=("penalty_p85_min", "median"),
    median_penalty_p95_min=("penalty_p95_min", "median"),
    max_penalty_p85_min=("penalty_p85_min", "max"),
    max_penalty_p95_min=("penalty_p95_min", "max"),
    mean_ratio_p85_to_p50=("ratio_p85_to_p50", "mean"),
    mean_ratio_p95_to_p50=("ratio_p95_to_p50", "mean"),
    n_od_pairs=("origin_id", "count")
).reset_index()

traffic_penalty_by_date.to_csv(AUDIT_DIR / "traffic_penalty_summary_by_date.csv", index=False, encoding="utf-8-sig")
save_df(traffic_penalty_by_date, "Traffic Penalty Summary by Date", "audit_traffic")
display(traffic_penalty_by_date)

# ------------------------------------------------------------
# 10C.4 Traffic penalty visuals
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(["p85 - p50", "p95 - p50"], [
    traffic_penalty["penalty_p85_min"].mean(),
    traffic_penalty["penalty_p95_min"].mean()
])
ax.set_title("Visual Traffic Penalty 1. Rata-rata Tambahan Waktu Akibat Kemacetan", fontweight="bold")
ax.set_ylabel("Tambahan waktu tempuh rata-rata (menit)")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Visual Traffic Penalty 1 Rata-rata Tambahan Waktu Akibat Kemacetan")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(traffic_penalty_by_date["delivery_date"], traffic_penalty_by_date["mean_penalty_p85_min"], marker="o", label="p85 - p50")
ax.plot(traffic_penalty_by_date["delivery_date"], traffic_penalty_by_date["mean_penalty_p95_min"], marker="o", label="p95 - p50")
ax.set_title("Visual Traffic Penalty 2. Tambahan Waktu Rata-rata per Tanggal", fontweight="bold")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Tambahan waktu rata-rata (menit)")
ax.tick_params(axis="x", rotation=30)
ax.legend()
ax.grid(alpha=0.25)
save_fig(fig, "Visual Traffic Penalty 2 Tambahan Waktu Rata-rata per Tanggal")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(traffic_penalty["penalty_p85_min"].dropna(), bins=30, alpha=0.7, label="p85 - p50")
ax.hist(traffic_penalty["penalty_p95_min"].dropna(), bins=30, alpha=0.7, label="p95 - p50")
ax.set_title("Visual Traffic Penalty 3. Distribusi Tambahan Waktu OD Pairs", fontweight="bold")
ax.set_xlabel("Tambahan waktu tempuh (menit)")
ax.set_ylabel("Frekuensi")
ax.legend()
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Visual Traffic Penalty 3 Distribusi Tambahan Waktu OD Pairs")

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([
    traffic_penalty["travel_time_min_p50"].dropna(),
    traffic_penalty["travel_time_min_p85"].dropna(),
    traffic_penalty["travel_time_min_p95"].dropna()
], labels=["p50", "p85", "p95"])
ax.set_title("Visual Traffic Penalty 4. Boxplot Travel Time p50-p85-p95", fontweight="bold")
ax.set_ylabel("Travel time (menit)")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Visual Traffic Penalty 4 Boxplot Travel Time p50-p85-p95")

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(["p85/p50", "p95/p50"], [
    traffic_penalty["ratio_p85_to_p50"].mean(),
    traffic_penalty["ratio_p95_to_p50"].mean()
])
ax.set_title("Visual Traffic Penalty 5. Rasio Kenaikan Travel Time terhadap p50", fontweight="bold")
ax.set_ylabel("Rasio rata-rata")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Visual Traffic Penalty 5 Rasio Kenaikan Travel Time terhadap p50")

# Heatmap traffic penalty untuk sample date
sample_penalty_df = traffic_penalty[traffic_penalty["delivery_date"] == sample_date_used].copy()

if not sample_penalty_df.empty:
    sample_nodes_ids = nodes_sample["node_id"].astype(str).tolist()
    sample_p85_penalty_mat = sample_penalty_df.pivot(
        index="origin_id", columns="destination_id", values="penalty_p85_min"
    ).reindex(index=sample_nodes_ids, columns=sample_nodes_ids)

    sample_p95_penalty_mat = sample_penalty_df.pivot(
        index="origin_id", columns="destination_id", values="penalty_p95_min"
    ).reindex(index=sample_nodes_ids, columns=sample_nodes_ids)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(sample_p85_penalty_mat.fillna(0).values)
    ax.set_title(f"Visual Traffic Penalty 6. Heatmap Penalty p85-p50 — {sample_date_used}", fontweight="bold")
    ax.set_xlabel("Destination node index")
    ax.set_ylabel("Origin node index")
    fig.colorbar(im, ax=ax, label="Penalty (menit)")
    save_fig(fig, "Visual Traffic Penalty 6 Heatmap Penalty p85-p50 Sample")

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(sample_p95_penalty_mat.fillna(0).values)
    ax.set_title(f"Visual Traffic Penalty 7. Heatmap Penalty p95-p50 — {sample_date_used}", fontweight="bold")
    ax.set_xlabel("Destination node index")
    ax.set_ylabel("Origin node index")
    fig.colorbar(im, ax=ax, label="Penalty (menit)")
    save_fig(fig, "Visual Traffic Penalty 7 Heatmap Penalty p95-p50 Sample")

# ------------------------------------------------------------
# 10C.5 Vehicle-profile limitation audit
# ------------------------------------------------------------
vehicle_profile_audit = pd.DataFrame({
    "Aspek": [
        "Road-network source",
        "Routing engine",
        "Routing profile",
        "Vehicle approximation",
        "Suitability for small/medium box vehicle",
        "Truck-specific restriction",
        "Vehicle height restriction",
        "Vehicle width restriction",
        "Vehicle weight restriction",
        "Hazmat/commercial restriction",
        "Traffic congestion representation",
        "Traffic-light delay",
        "Traffic-signal count",
        "Intersection delay",
        "Recommended interpretation"
    ],
    "Status dalam Notebook": [
        "Covered",
        "Covered",
        "Covered",
        "Partially covered",
        "Reasonable approximation",
        "Not explicitly covered",
        "Not explicitly covered",
        "Not explicitly covered",
        "Not explicitly covered",
        "Not explicitly covered",
        "Covered as multiplier",
        "Not explicitly separated",
        "Not explicitly calculated",
        "Not explicitly calculated",
        "Use as road-network driving approximation with traffic multiplier"
    ],
    "Keterangan": [
        "Menggunakan saved OSRM road-network matrix.",
        "Matriks berasal dari OSRM Table Service output yang disimpan.",
        "Profile yang digunakan adalah driving.",
        "Mobil box direpresentasikan sebagai kendaraan distribusi jalan raya umum.",
        "Aman sebagai pendekatan jika mobil box mengikuti jalan umum dan tidak memerlukan truck restriction khusus.",
        "Larangan khusus truk/kendaraan barang belum dimodelkan.",
        "Batas tinggi kendaraan belum menjadi constraint.",
        "Batas lebar kendaraan belum menjadi constraint.",
        "Batas berat kendaraan pada ruas jalan belum menjadi constraint OSRM.",
        "Batasan kendaraan komersial spesifik belum dimasukkan.",
        "Kemacetan direpresentasikan oleh TomTom-based multiplier p50, p85, p95.",
        "Delay lampu merah tidak dihitung sebagai komponen terpisah.",
        "Jumlah traffic signal pada rute tidak diekstrak.",
        "Delay persimpangan tidak dihitung secara eksplisit.",
        "Klaim metodologis perlu ditulis sebagai OSRM driving road-network matrix + TomTom-based traffic multiplier."
    ],
    "Implikasi terhadap Disertasi": [
        "Mendukung klaim road-network-based routing.",
        "Mendukung pembentukan distance dan duration matrix.",
        "Sesuai untuk kendaraan bermotor umum, tetapi belum truck-specific.",
        "Perlu dijelaskan sebagai pendekatan kendaraan distribusi, bukan routing truk penuh.",
        "Dapat diterima untuk konteks distribusi perkotaan mobil box kecil/medium.",
        "Masuk ke keterbatasan penelitian/future work.",
        "Masuk ke keterbatasan penelitian/future work.",
        "Masuk ke keterbatasan penelitian/future work.",
        "Masuk ke keterbatasan penelitian/future work.",
        "Masuk ke keterbatasan penelitian/future work.",
        "Mendukung analisis skenario p50, p85, p95.",
        "Jangan diklaim sebagai model signal-delay eksplisit.",
        "Jangan diklaim sebagai evaluasi jumlah lampu merah.",
        "Jangan diklaim sebagai intersection-delay model.",
        "Kalimat metodologis harus transparan dan tidak overclaim."
    ]
})
vehicle_profile_audit.to_csv(AUDIT_DIR / "vehicle_profile_limitation_audit.csv", index=False, encoding="utf-8-sig")
save_df(vehicle_profile_audit, "Vehicle-Profile Limitation Audit", "audit_vehicle")
display(vehicle_profile_audit)

# ------------------------------------------------------------
# 10C.6 Road-network and traffic component summary table
# ------------------------------------------------------------
component_summary = pd.DataFrame({
    "Komponen": [
        "OSRM distance matrix",
        "OSRM duration matrix",
        "TomTom-based multiplier",
        "Travel-time p50",
        "Travel-time p85",
        "Travel-time p95",
        "Traffic penalty p85",
        "Traffic penalty p95",
        "Vehicle profile",
        "Traffic signal delay"
    ],
    "Rumus / Sumber": [
        "Saved OSRM distance matrix",
        "Saved OSRM duration matrix",
        "Saved TomTom multiplier",
        "OSRM duration × multiplier p50",
        "OSRM duration × multiplier p85",
        "OSRM duration × multiplier p95",
        "travel_time_p85 − travel_time_p50",
        "travel_time_p95 − travel_time_p50",
        "OSRM driving profile",
        "Not explicitly modeled"
    ],
    "Fungsi dalam Penelitian": [
        "Mengukur jarak jaringan jalan antar node.",
        "Menjadi base travel time sebelum penyesuaian traffic.",
        "Merepresentasikan variasi kemacetan.",
        "Kondisi perjalanan normal/median.",
        "Kondisi perjalanan padat.",
        "Kondisi perjalanan sangat padat.",
        "Mengukur tambahan waktu akibat skenario traffic p85.",
        "Mengukur tambahan waktu akibat skenario traffic p95.",
        "Merepresentasikan kendaraan bermotor jalan raya umum.",
        "Dicatat sebagai keterbatasan, bukan komponen model eksplisit."
    ]
})
component_summary.to_csv(AUDIT_DIR / "road_network_and_traffic_component_summary.csv", index=False, encoding="utf-8-sig")
save_df(component_summary, "Road-Network and Traffic Component Summary", "audit_traffic")
display(component_summary)

display(Markdown("### Traffic penalty dan vehicle-profile audit selesai. Semua tabel dan visual tersimpan ke output folder."))

In [ ]:
# ============================================================
# CELL 11 — KUANTIL WAKTU TEMPUH DAN SKENARIO EKSPERIMEN
# ============================================================

t49 = pd.DataFrame({
    "Kuantil": ["p50", "p85", "p95"],
    "Interpretasi Operasional": ["Kondisi perjalanan normal", "Kondisi perjalanan padat", "Kondisi perjalanan sangat padat"],
    "Peran dalam Evaluasi": ["Acuan median/kondisi umum", "Menguji dampak peningkatan waktu tempuh", "Menguji performa pada kondisi konservatif tinggi"]
})
save_df(t49, "Tabel 4.9 Interpretasi Kuantil Waktu Tempuh", "Tabel_4_9")

quantile_results = pd.DataFrame({
    "Matriks Waktu Tempuh": ["Matriks p50", "Matriks p85", "Matriks p95"],
    "Kuantil": ["p50", "p85", "p95"],
    "Total Travel Time": [31700, 36464, 41434],
    "Perubahan terhadap p50": [0.00, 15.03, 30.71],
    "Total Lateness": [19740.20, 20990.50, 18809.20],
    "Perubahan Lateness terhadap p50": [0.00, 6.33, -4.72],
    "OTD": [95.60, 95.29, 95.58],
    "Perubahan OTD terhadap p50": [0.00, -0.31, -0.02]
})

save_df(quantile_results[["Matriks Waktu Tempuh","Kuantil","Total Travel Time","Perubahan terhadap p50"]],
        "Tabel 4.10 Magnitudo Risiko Waktu Tempuh Berdasarkan Matriks Kuantil", "Tabel_4_10")
save_df(quantile_results[["Matriks Waktu Tempuh","Kuantil","Total Lateness","Perubahan Lateness terhadap p50","OTD","Perubahan OTD terhadap p50"]],
        "Tabel 4.11 Dampak Penggunaan Matriks Kuantil Tetap terhadap Lateness", "Tabel_4_11")

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(quantile_results["Kuantil"], quantile_results["Total Travel Time"])
ax.set_title("Gambar 4.7 Magnitudo Risiko Waktu Tempuh Kuantil", fontweight="bold")
ax.set_ylabel("Total Travel Time (menit)")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Gambar 4.7 Magnitudo Risiko Waktu Tempuh Kuantil")

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(quantile_results["Kuantil"], quantile_results["Total Lateness"])
ax.set_title("Gambar 4.8 Dampak Penggunaan Matriks Kuantil Tetap terhadap Lateness", fontweight="bold")
ax.set_ylabel("Total Lateness (menit)")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Gambar 4.8 Dampak Penggunaan Matriks Kuantil Tetap terhadap Lateness")

scenario_design = pd.DataFrame({
    "Skenario": ["S0", "S1", "S2", "S3", "S4", "S5"],
    "Nama Skenario": ["VRPTW-1D Static Baseline", "MCVRPTW-TD p50 Reference", "Fixed p85 Scenario", "Fixed p95 Scenario", "QRG-ALNS-ILS Proposed Model", "Diagnostic Feedback Scenario"],
    "Karakteristik Utama": [
        "Kapasitas satu dimensi dan waktu tempuh statis",
        "Kapasitas berat-volume dan matriks waktu tempuh p50",
        "Kapasitas berat-volume dan p85 tetap",
        "Kapasitas berat-volume dan p95 tetap",
        "Risk-guided search, risk escalation, risk-aware insertion, dual-capacity checking",
        "Freeze served stops dan re-sequence remaining stops"
    ],
    "Tujuan Evaluasi": [
        "Baseline awal",
        "Dampak kapasitas dua dimensi dan p50",
        "Dampak fixed p85",
        "Dampak fixed p95",
        "Kontribusi algoritma usulan",
        "Diagnostic feedback dan re-planning adaptif terbatas"
    ]
})
save_df(scenario_design, "Tabel 4.12 Desain Skenario Eksperimen", "Tabel_4_12")

availability = pd.DataFrame({
    "Skenario": ["S0", "S1", "S2", "S3", "S4", "S5"],
    "Status Hasil": ["Tersedia", "Tersedia", "Tersedia", "Tersedia", "Tersedia", "Mekanisme tersedia, pengujian penuh dikembangkan bertahap"],
    "Fokus Analisis": [
        "Baseline statis dan kapasitas satu dimensi",
        "Dampak kapasitas berat-volume dan waktu tempuh p50",
        "Dampak fixed p85 sebagai skenario kontrol",
        "Dampak fixed p95 sebagai skenario kontrol",
        "Evaluasi model usulan QRG-ALNS-ILS",
        "Diagnostic feedback dan re-planning adaptif terbatas"
    ]
})
save_df(availability, "Tabel 4.13 Ketersediaan Hasil dan Fokus Analisis", "Tabel_4_13")

In [ ]:
# ============================================================
# CELL 12 — HASIL EKSPERIMEN FINAL S0–S4
# ============================================================

scenario_results = pd.DataFrame({
    "Skenario": ["S0", "S1", "S2", "S3", "S4"],
    "Total Distance (km)": [17437.84, 17768.44, 17537.45, 17565.87, 18496.37],
    "Total Travel Time (menit)": [28561, 31700, 36464, 41434, 33559],
    "Total Lateness (menit)": [16130.80, 19740.20, 20990.50, 18809.20, 14101.50],
    "OTD (%)": [96.09, 95.60, 95.29, 95.58, 96.97],
})

save_df(scenario_results, "Tabel 4.16 Hasil Agregat Perbandingan Skenario S0-S4", "Tabel_4_16")

s4 = scenario_results[scenario_results["Skenario"]=="S4"].iloc[0]
impr_rows = []
for comp in ["S0", "S1", "S2", "S3"]:
    r = scenario_results[scenario_results["Skenario"]==comp].iloc[0]
    impr_rows.append({
        "Perbandingan": f"S4 terhadap {comp}",
        "Perubahan Distance": pct_change(s4["Total Distance (km)"], r["Total Distance (km)"]),
        "Perubahan Travel Time": pct_change(s4["Total Travel Time (menit)"], r["Total Travel Time (menit)"]),
        "Penurunan Lateness": improvement_lower_is_better(s4["Total Lateness (menit)"], r["Total Lateness (menit)"]),
        "Perubahan OTD": s4["OTD (%)"] - r["OTD (%)"]
    })
t417 = pd.DataFrame(impr_rows)
save_df(t417, "Tabel 4.17 Magnitudo Perbaikan S4 terhadap Skenario Pembanding", "Tabel_4_17")
save_df(t417, "Tabel 4.22 Magnitudo Perbaikan S4 terhadap Skenario Pembanding", "Tabel_4_22")

s0 = scenario_results[scenario_results["Skenario"]=="S0"].iloc[0]
s1 = scenario_results[scenario_results["Skenario"]=="S1"].iloc[0]

t418 = pd.DataFrame({
    "Metrik": ["Total Distance", "Total Travel Time", "Total Lateness", "OTD"],
    "S0": [s0["Total Distance (km)"], s0["Total Travel Time (menit)"], s0["Total Lateness (menit)"], s0["OTD (%)"]],
    "S1": [s1["Total Distance (km)"], s1["Total Travel Time (menit)"], s1["Total Lateness (menit)"], s1["OTD (%)"]],
    "Perubahan": [
        pct_change(s1["Total Distance (km)"], s0["Total Distance (km)"]),
        pct_change(s1["Total Travel Time (menit)"], s0["Total Travel Time (menit)"]),
        pct_change(s1["Total Lateness (menit)"], s0["Total Lateness (menit)"]),
        s1["OTD (%)"] - s0["OTD (%)"]
    ]
})
save_df(t418, "Tabel 4.18 Perubahan Performa S1 terhadap S0", "Tabel_4_18")

t419 = scenario_results[scenario_results["Skenario"].isin(["S1","S2","S3"])].copy()
save_df(t419, "Tabel 4.19 Hasil Perbandingan S1, S2, dan S3", "Tabel_4_19")

rows = []
for sk in ["S2","S3"]:
    r = scenario_results[scenario_results["Skenario"]==sk].iloc[0]
    rows.append({
        "Skenario": sk,
        "Perubahan Distance terhadap S1": pct_change(r["Total Distance (km)"], s1["Total Distance (km)"]),
        "Perubahan Travel Time terhadap S1": pct_change(r["Total Travel Time (menit)"], s1["Total Travel Time (menit)"]),
        "Perubahan Lateness terhadap S1": pct_change(r["Total Lateness (menit)"], s1["Total Lateness (menit)"]),
        "Perubahan OTD terhadap S1": r["OTD (%)"] - s1["OTD (%)"]
    })
t420 = pd.DataFrame(rows)
save_df(t420, "Tabel 4.20 Perubahan Performa S2 dan S3 terhadap S1", "Tabel_4_20")
save_df(scenario_results, "Tabel 4.21 Performa S4 terhadap Seluruh Skenario Pembanding", "Tabel_4_21")

rank_df = scenario_results.copy()
rank_df["Rank Lateness"] = rank_df["Total Lateness (menit)"].rank(method="min")
rank_df["Rank OTD"] = rank_df["OTD (%)"].rank(ascending=False, method="min")
rank_df["Rank Distance"] = rank_df["Total Distance (km)"].rank(method="min")
rank_df["Rank Travel Time"] = rank_df["Total Travel Time (menit)"].rank(method="min")
save_df(rank_df, "Tabel 4.24 Peringkat Performa Skenario Berdasarkan Metrik Utama", "Tabel_4_24")

display(scenario_results)

In [ ]:
# ============================================================
# CELL 13 — GAMBAR HASIL EKSPERIMEN DAN TRADE-OFF
# ============================================================

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(scenario_results["Skenario"], scenario_results["Total Lateness (menit)"])
ax.set_title("Gambar 4.12 Perbandingan Total Lateness S0-S4", fontweight="bold")
ax.set_ylabel("Total Lateness (menit)")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Gambar 4.12 Perbandingan Total Lateness S0-S4")

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(scenario_results["Skenario"], scenario_results["OTD (%)"])
ax.set_title("Gambar 4.13 Perbandingan On-Time Delivery S0-S4", fontweight="bold")
ax.set_ylabel("OTD (%)")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Gambar 4.13 Perbandingan On-Time Delivery S0-S4")

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(scenario_results["Skenario"], scenario_results["Total Travel Time (menit)"])
ax.set_title("Gambar 4.14 Perbandingan Total Travel Time S0-S4", fontweight="bold")
ax.set_ylabel("Total Travel Time (menit)")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Gambar 4.14 Perbandingan Total Travel Time S0-S4")

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(t418["Metrik"], t418["Perubahan"])
ax.axhline(0, linewidth=1)
ax.set_title("Gambar 4.17 Perubahan Performa S1 terhadap S0", fontweight="bold")
ax.set_ylabel("Perubahan (%) / pp")
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Gambar 4.17 Perubahan Performa S1 terhadap S0")

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(t417["Perbandingan"], t417["Penurunan Lateness"])
ax.set_title("Gambar 4.20 Penurunan Total Lateness S4 terhadap Skenario Pembanding", fontweight="bold")
ax.set_ylabel("Penurunan Lateness (%)")
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", alpha=0.25)
save_fig(fig, "Gambar 4.20 Penurunan Total Lateness S4 terhadap Skenario Pembanding")

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(t417["Perubahan Distance"], t417["Penurunan Lateness"], s=90)
for _, r in t417.iterrows():
    label = r["Perbandingan"].replace("S4 terhadap ", "")
    ax.annotate(label, (r["Perubahan Distance"], r["Penurunan Lateness"]), xytext=(5,5), textcoords="offset points")
ax.set_title("Gambar 4.21 Trade-off Tambahan Jarak dan Penurunan Lateness pada S4", fontweight="bold")
ax.set_xlabel("Perubahan Distance S4 (%)")
ax.set_ylabel("Penurunan Lateness S4 (%)")
ax.grid(alpha=0.25)
save_fig(fig, "Gambar 4.21 Trade-off Tambahan Jarak dan Penurunan Lateness pada S4")

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(scenario_results["Total Distance (km)"], scenario_results["Total Lateness (menit)"], s=90)
for _, r in scenario_results.iterrows():
    ax.annotate(r["Skenario"], (r["Total Distance (km)"], r["Total Lateness (menit)"]), xytext=(5,5), textcoords="offset points")
ax.set_title("Gambar 4.24 Scatter Plot Reliability-Efficiency Trade-Off S0-S4", fontweight="bold")
ax.set_xlabel("Total Distance (km)")
ax.set_ylabel("Total Lateness (menit)")
ax.grid(alpha=0.25)
save_fig(fig, "Gambar 4.24 Scatter Plot Reliability-Efficiency Trade-Off S0-S4")

In [ ]:
# ============================================================
# CELL 14 — TABEL MEKANISME ALGORITMA, KAPASITAS, DAN S5
# ============================================================

t423 = pd.DataFrame({
    "Komponen QRG-ALNS-ILS": ["Lateness-priority removal", "Risk-aware insertion", "Dual-capacity feasibility checking", "Local search improvement"],
    "Peran": [
        "Membongkar pelanggan atau segmen yang berkontribusi besar terhadap keterlambatan",
        "Menyisipkan pelanggan berdasarkan tambahan jarak, waktu tempuh, lateness, risiko, dan kapasitas",
        "Menolak kandidat yang melanggar batas berat atau volume",
        "Memperbaiki urutan pelanggan melalui relocate, swap, dan 2-opt"
    ],
    "Dampak terhadap Reliabilitas": [
        "Mengurangi sumber keterlambatan utama",
        "Menghindari penyisipan yang memperbesar risiko lateness",
        "Menjaga solusi tetap feasible terhadap pemuatan nyata",
        "Menurunkan lateness melalui perbaikan urutan lokal"
    ]
})
save_df(t423, "Tabel 4.23 Komponen Utama QRG-ALNS-ILS dan Perannya", "Tabel_4_23")

t425 = pd.DataFrame({
    "Indikator": [
        "Total berat pengiriman", "Total volume pengiriman", "Densitas agregat",
        "Proporsi bulky / volume-dominant", "Proporsi heavy / weight-dominant",
        "Korelasi berat-volume order-level", "Korelasi berat-volume customer-day stop"
    ],
    "Nilai": [
        f"{total_weight:,.2f} kg".replace(",", "X").replace(".", ",").replace("X", "."),
        f"{total_volume_m3:,.2f} m³".replace(",", "X").replace(".", ",").replace("X", "."),
        f"{agg_density:,.2f} kg/m³".replace(",", "X").replace(".", ",").replace("X", "."),
        t44.loc[0,"Proporsi"], t44.loc[1,"Proporsi"],
        f"{t43.loc[0,'Korelasi Berat-Volume']:.3f}",
        f"{t43.loc[1,'Korelasi Berat-Volume']:.3f}"
    ],
    "Implikasi": [
        "Kapasitas berat tetap relevan",
        "Kapasitas volume sangat penting",
        "Menjadi threshold klasifikasi bulky-heavy",
        "Risiko cubed-out tinggi",
        "Risiko overweight tetap perlu dijaga",
        "Berat-volume terkait kuat tetapi tidak identik",
        "Agregasi tetap mempertahankan kebutuhan kapasitas dua dimensi"
    ]
})
save_df(t425, "Tabel 4.25 Ringkasan Indikator Agregat Berat-Volume", "Tabel_4_25")

t426 = pd.DataFrame({
    "Tahap": ["Hitung total berat rute", "Hitung total volume rute", "Bandingkan dengan kapasitas", "Keputusan"],
    "Pemeriksaan": ["Σw_i pada rute", "Σv_i pada rute", "Σw_i ≤ QW dan Σv_i ≤ QV", "Feasible jika kedua kendala terpenuhi"],
    "Makna": ["Menjaga batas berat kendaraan", "Menjaga batas ruang muat", "Kelayakan tidak boleh hanya satu dimensi", "Jika salah satu melanggar, kandidat ditolak"]
})
save_df(t426, "Tabel 4.26 Mekanisme Dual-Capacity Feasibility Checking", "Tabel_4_26")

t427 = pd.DataFrame({
    "Elemen": ["Freeze served stops", "Re-sequence remaining stops", "Risk trigger", "Stability penalty", "Diagnostic feedback"],
    "Penjelasan": [
        "Pelanggan yang sudah dikunjungi dikunci dan tidak diubah kembali",
        "Pelanggan yang belum dikunjungi dapat diurutkan ulang",
        "Re-planning dilakukan ketika risiko keterlambatan melewati ambang tertentu",
        "Perubahan rute diberi penalti agar tidak terlalu drastis",
        "Hasil evaluasi risiko dikembalikan ke proses routing"
    ],
    "Implikasi Operasional": [
        "Menjaga konsistensi dengan kondisi lapangan yang sudah terjadi",
        "Memberi ruang perbaikan pada bagian rute yang masih dapat dikendalikan",
        "Mencegah perubahan rute yang tidak perlu",
        "Menjaga rute tetap dapat diterima dan dipahami oleh pengemudi",
        "Menutup siklus Digital Twin dari evaluasi ke tindakan korektif"
    ]
})
save_df(t427, "Tabel 4.27 Prinsip Operasional Diagnostic Feedback pada S5", "Tabel_4_27")

t428 = pd.DataFrame({
    "Tahap": ["Rencana awal", "Monitoring berjalan", "Trigger risiko", "Freeze", "Re-sequence", "Evaluasi stabilitas", "Output"],
    "Kondisi": [
        "Rute S4 telah dibuat untuk pelanggan 1 sampai 10",
        "Pada pukul 11:00 terdeteksi deviasi waktu tempuh 15 menit",
        "Lpred melewati ambang risiko θ2",
        "Pelanggan 1 sampai 5 sudah dilayani",
        "Pelanggan 6 sampai 10 belum dilayani",
        "Rute baru dibandingkan dengan rute awal",
        "Rute pembaruan terbatas dihasilkan"
    ],
    "Tindakan Sistem": [
        "Rute digunakan sebagai rencana operasional awal",
        "Sistem menghitung potensi keterlambatan terhadap pelanggan tersisa",
        "Diagnostic feedback diaktifkan",
        "Posisi pelanggan 1 sampai 5 dikunci dan tidak diubah",
        "Sistem menyusun ulang urutan pelanggan 6 sampai 10",
        "Perubahan yang terlalu ekstrem diberi stability penalty",
        "Sistem memberikan rekomendasi rute untuk meminimalkan dampak keterlambatan"
    ]
})
save_df(t428, "Tabel 4.28 Ilustrasi Walkthrough Diagnostic Feedback pada S5", "Tabel_4_28")

In [ ]:
# ============================================================
# CELL 15 — DIAGRAM ALUR DAN MEKANISME
# ============================================================

save_flow("Gambar 3.1 Peta Tahapan Penelitian dan Letak Kebaruan", [
    ("Identifikasi Masalah", "Distribusi kosmetik\nTW, kapasitas, risiko"),
    ("Kajian Pustaka", "DT logistics, VRPTW,\nTDVRP, MDCVRP"),
    ("Data & Pra-Proses", "Order, koordinat,\nberat-volume, TW"),
    ("Matriks Kuantil", "Road-network\np50, p85, p95"),
    ("QRG-ALNS-ILS", "Risk-guided\ndestroy-repair"),
    ("Evaluasi & Feedback", "S0-S5\ntrade-off & S5")
], "Gambar_3_1_Peta_Tahapan_Penelitian_dan_Letak_Kebaruan.png", figsize=(14,5))

save_flow("Gambar 3.2 Alur Algoritma QRG-ALNS-ILS", [
    ("Solusi Awal", "Feasible terhadap\nTW & kapasitas"),
    ("Destroy", "Random/worst/\nlateness-priority"),
    ("Risk-aware Repair", "Insertion score\nberbasis risiko"),
    ("Dual-Capacity", "Cek berat dan\nvolume simultan"),
    ("Local Search", "Relocate, swap,\n2-opt"),
    ("Evaluasi", "Distance, travel time,\nlateness, OTD")
], "Gambar_3_2_Alur_Algoritma_QRG_ALNS_ILS.png", figsize=(14,5))

save_flow("Gambar 3.3 Digital Twin-Based Routing Framework", [
    ("Data Layer", "Pelanggan, order,\nTW, kapasitas"),
    ("Travel-Time Layer", "Matriks jarak &\nwaktu kuantil"),
    ("Optimization Layer", "S0-S4\nQRG-ALNS-ILS"),
    ("Risk Evaluation", "Lateness, OTD,\nrisk escalation"),
    ("Feedback Layer", "Freeze served &\nre-sequence")
], "Gambar_3_3_Digital_Twin_Based_Routing_Framework.png", figsize=(13,5))

save_flow("Gambar 3.4 Diagnostic Feedback / Freeze and Re-sequence", [
    ("Monitoring", "Status kunjungan\n& deviasi waktu"),
    ("Risk Trigger", "Lpred > θ2"),
    ("Freeze", "Stop yang sudah\ndilayani dikunci"),
    ("Re-sequence", "Stop tersisa\ndiurutkan ulang"),
    ("Updated Route", "Rute pembaruan\nadaptif terbatas")
], "Gambar_3_4_Diagnostic_Feedback_Freeze_and_Re_sequence.png", figsize=(12,5))

save_flow("Gambar 4.9 Alur Evaluasi Skenario Eksperimen", [
    ("S0", "Static baseline\n1D capacity"),
    ("S1", "TD p50\n2D capacity"),
    ("S2", "Fixed p85"),
    ("S3", "Fixed p95"),
    ("S4", "QRG-ALNS-ILS"),
    ("S5", "Diagnostic feedback")
], "Gambar_4_9_Alur_Evaluasi_Skenario_Eksperimen.png", figsize=(14,5))

save_flow("Gambar 4.10 Peta Ablation Study dan Kontribusi Komponen", [
    ("Baseline", "S0"),
    ("+ Realisme Model", "S1\n2D + TD p50"),
    ("+ Fixed Quantile", "S2/S3\np85/p95"),
    ("+ Risk Guidance", "S4\nQRG-ALNS-ILS"),
    ("+ Feedback", "S5\nFreeze & re-sequence")
], "Gambar_4_10_Peta_Ablation_Study_dan_Kontribusi_Komponen.png", figsize=(12,5))

save_flow("Gambar 4.16 Mekanisme Counter-Intuitive pada S1", [
    ("S0", "Baseline tampak\nefisien"),
    ("S1", "2D capacity\n+ TD p50"),
    ("Constraint Tightening", "Slack rute\nmenyempit"),
    ("Knock-on Effect", "Keterlambatan\nberantai"),
    ("Lateness Naik", "+22,38%\nvs S0")
], "Gambar_4_16_Mekanisme_Counter_Intuitive_pada_S1.png", figsize=(13,5))

save_flow("Gambar 4.18 Penjelasan Constraint Tightening pada S1", [
    ("Kapasitas 2D", "Berat dan volume\nwajib feasible"),
    ("TD p50", "Arrival time\nlebih realistis"),
    ("Slack Menyempit", "Opsi insertion\nlebih terbatas"),
    ("Akumulasi Lateness", "Keterlambatan\nmeningkat")
], "Gambar_4_18_Penjelasan_Constraint_Tightening_pada_S1.png", figsize=(11,5))

save_flow("Gambar 4.19 Keterbatasan Fixed Quantile dan Posisi S4", [
    ("S1 p50", "Normal tetapi\nbelum risk-guided"),
    ("S2 p85", "Konservatif sedang\nnamun kaku"),
    ("S3 p95", "Sangat konservatif\ntravel time tinggi"),
    ("S4", "Adaptive quantile\nrisk-guided search")
], "Gambar_4_19_Keterbatasan_Fixed_Quantile_dan_Posisi_S4.png", figsize=(12,5))

save_flow("Gambar 4.22 Posisi Kontribusi S4 dalam Rangkaian Skenario", [
    ("S0", "Optimisme\nbaseline"),
    ("S1", "Realisme model\nlateness naik"),
    ("S2/S3", "Fixed quantile\nkurang adaptif"),
    ("S4", "Risk-guided adaptive\nreliability terbaik")
], "Gambar_4_22_Posisi_Kontribusi_S4_dalam_Rangkaian_Skenario.png", figsize=(12,5))

save_flow("Gambar 4.23 Mekanisme Internal QRG-ALNS-ILS pada S4", [
    ("Diagnose", "Identifikasi\nlateness risk"),
    ("Destroy", "Lateness-priority\nremoval"),
    ("Repair", "Risk-aware\ninsertion"),
    ("Feasibility", "Dual-capacity\nchecking"),
    ("Improve", "Local search"),
    ("Select", "Solusi lebih\nreliabel")
], "Gambar_4_23_Mekanisme_Internal_QRG_ALNS_ILS_pada_S4.png", figsize=(14,5))

save_flow("Gambar 4.25 Mekanisme Dual-Capacity Feasibility Checking pada S4", [
    ("Candidate Insertion", "Pelanggan\nkandidat"),
    ("Update Load", "Berat + volume"),
    ("Weight Check", "Σw ≤ QW"),
    ("Volume Check", "Σv ≤ QV"),
    ("Feasible / Reject", "Layak jika\nkeduanya terpenuhi")
], "Gambar_4_25_Mekanisme_Dual_Capacity_Feasibility_Checking_pada_S4.png", figsize=(13,5))

save_flow("Gambar 4.26 Alur Diagnostic Feedback pada S5", [
    ("Rute Awal S4", "QRG-ALNS-ILS"),
    ("Monitoring", "Deviasi waktu\n& status stop"),
    ("Risk Trigger", "Lpred > θ2"),
    ("Freeze", "Stop dilayani\ndikunci"),
    ("Re-sequence", "Stop tersisa\ndioptimasi ulang"),
    ("Feedback Layer", "Tindakan\nkorektif")
], "Gambar_4_26_Alur_Diagnostic_Feedback_pada_S5.png", figsize=(14,5))

print("Diagram selesai.")

In [ ]:
# ============================================================
# CELL 16 — SIMULASI DIAGNOSTIC FEEDBACK S5
# ============================================================

initial_route = [1,2,3,4,5,6,7,8,9,10]
served = [1,2,3,4,5]
remaining = [6,7,8,9,10]
Lpred = 22
theta2 = 15

if Lpred > theta2:
    updated_route = served + [6,8,7,9,10]
    feedback_active = True
else:
    updated_route = initial_route
    feedback_active = False

s5_demo = pd.DataFrame({
    "Komponen": ["Initial Route", "Served / Frozen", "Remaining", "Lpred", "theta2", "Feedback Active", "Updated Route"],
    "Nilai": [str(initial_route), str(served), str(remaining), Lpred, theta2, feedback_active, str(updated_route)]
})
save_df(s5_demo, "Audit Simulasi S5 Diagnostic Feedback", "audit_s5")
display(s5_demo)

In [ ]:
# ============================================================
# CELL 17 — RENCANA LANJUTAN DAN SLOT UJI SIGNIFIKANSI
# ============================================================

t51 = pd.DataFrame({
    "No.": [1,2,3,4,5,6,7],
    "Rencana Lanjutan": [
        "Finalisasi hasil S0–S4 per instance",
        "Robustness lanjutan berdasarkan tertile instance",
        "Analisis sensitivitas parameter",
        "Analisis utilisasi kapasitas per rute",
        "Pengujian S5 dengan skenario deviasi",
        "Penyusunan visualisasi Digital Twin-Based Routing",
        "Finalisasi dokumen disertasi"
    ],
    "Tujuan": [
        "Memperkuat bukti empiris model usulan",
        "Menguji konsistensi S4 pada skala kecil, sedang, dan besar",
        "Menguji stabilitas model terhadap perubahan bobot dan threshold",
        "Memperkuat pembuktian kapasitas berat-volume",
        "Memvalidasi diagnostic feedback secara lebih kuat",
        "Menunjukkan integrasi data, optimasi, evaluasi risiko, dan feedback",
        "Menyelaraskan Bab 1 sampai Bab 5"
    ],
    "Output yang Diharapkan": [
        "Rekap performa per instance harian",
        "Tabel dan grafik performa per kelompok instance",
        "Hasil sensitivitas bobot, kuantil, dan risk escalation",
        "Rekap weight utilization dan volume utilization per kendaraan",
        "Re-planning gain dan stability penalty",
        "Dashboard atau prototype visualisasi",
        "Naskah disertasi yang konsisten dan siap diuji"
    ]
})
save_df(t51, "Tabel 5.1 Rencana Lanjutan Penelitian", "Tabel_5_1")

FULL_RESULT_FILE = Path("/content/route_results_per_instance_full.csv")

if FULL_RESULT_FILE.exists():
    full_res = pd.read_csv(FULL_RESULT_FILE)
    s1_l = full_res[full_res["scenario"]=="S1"].sort_values("delivery_date")["total_lateness_min"].values
    s4_l = full_res[full_res["scenario"]=="S4"].sort_values("delivery_date")["total_lateness_min"].values

    try:
        from scipy.stats import ttest_rel, wilcoxon
        t_stat, p_t = ttest_rel(s1_l, s4_l)
        try:
            w_stat, p_w = wilcoxon(s1_l, s4_l)
        except Exception:
            w_stat, p_w = np.nan, np.nan

        sig_df = pd.DataFrame({
            "Uji": ["Paired t-test", "Wilcoxon signed-rank"],
            "Statistik": [t_stat, w_stat],
            "p-value": [p_t, p_w],
            "Interpretasi": ["Signifikan jika p < 0,05", "Non-parametrik; signifikan jika p < 0,05"]
        })
        save_df(sig_df, "Uji Signifikansi S1 vs S4 Berdasarkan Per-Instance", "uji_signifikansi")
        display(sig_df)
    except Exception as e:
        print("Uji signifikansi tidak dapat dijalankan:", e)
else:
    note = pd.DataFrame({
        "Catatan": [
            "File route_results_per_instance_full.csv belum tersedia.",
            "Jika file hasil per-instance tersedia, unggah file tersebut untuk mengaktifkan uji signifikansi otomatis.",
            "Kolom minimal: delivery_date, scenario, total_lateness_min, otd_pct, total_distance_km, total_travel_time_min."
        ]
    })
    save_df(note, "Catatan Uji Signifikansi Per-Instance", "catatan")
    display(note)

## Preview Output di Notebook

Bagian berikut menampilkan hasil yang sudah dibuat langsung di Colab. Output tetap disimpan ke folder dan ZIP akhir, tetapi tabel dan gambar juga muncul sebagai preview di bawah cell.

In [ ]:
# ============================================================
# CELL 18 — PREVIEW OUTPUT DI COLAB
# ============================================================

from IPython.display import display, Image, Markdown

def preview_images_from_folder(folder, title, max_items=None):
    folder = Path(folder)
    files = sorted(list(folder.glob("*.png")))
    if max_items is not None:
        files = files[:max_items]

    display(Markdown(f"### {title}"))
    display(Markdown(f"Jumlah file PNG: **{len(files)}**"))

    for idx, p in enumerate(files, start=1):
        display(Markdown(f"**{idx}. {p.name}**"))
        display(Image(filename=str(p)))

def preview_csv_tables_from_folder(folder, title, max_items=None, max_rows=20):
    folder = Path(folder)
    files = sorted(list(folder.glob("*.csv")))
    if max_items is not None:
        files = files[:max_items]

    display(Markdown(f"### {title}"))
    display(Markdown(f"Jumlah file CSV: **{len(files)}**"))

    for idx, p in enumerate(files, start=1):
        display(Markdown(f"**{idx}. {p.name}**"))
        try:
            df_preview = pd.read_csv(p)
            display(df_preview.head(max_rows))
        except Exception as e:
            display(Markdown(f"Gagal membaca file: `{e}`"))

display(Markdown("## Ringkasan Output yang Dibuat"))

summary_preview = pd.DataFrame({
    "Komponen": ["CSV tables", "PNG tables", "PNG figures", "Audit outputs"],
    "Jumlah": [
        len(list(TABLE_DIR.glob("*.csv"))),
        len(list(TABLE_IMG_DIR.glob("*.png"))),
        len(list(FIG_DIR.glob("*.png"))),
        len(list(AUDIT_DIR.glob("*")))
    ]
})
display(summary_preview)

preview_csv_tables_from_folder(TABLE_DIR, "Preview Tabel CSV", max_items=None, max_rows=15)
preview_images_from_folder(TABLE_IMG_DIR, "Preview Tabel dalam Format Gambar", max_items=None)
preview_images_from_folder(FIG_DIR, "Preview Gambar dan Diagram", max_items=None)

display(Markdown("### Preview Audit Output"))
audit_files = sorted(list(AUDIT_DIR.glob("*.csv")))
display(Markdown(f"Jumlah audit CSV: **{len(audit_files)}**"))
for idx, p in enumerate(audit_files, start=1):
    display(Markdown(f"**{idx}. {p.name}**"))
    try:
        display(pd.read_csv(p).head(20))
    except Exception as e:
        display(Markdown(f"Gagal membaca file: `{e}`"))

In [ ]:
# ============================================================
# CELL 19 — MANIFEST OUTPUT DAN ZIP AKHIR
# ============================================================

manifest = []
for folder, label in [
    (TABLE_DIR, "table_csv"),
    (TABLE_IMG_DIR, "table_png"),
    (FIG_DIR, "figure_png"),
    (AUDIT_DIR, "audit_output"),
]:
    for p in sorted(folder.glob("*")):
        if p.is_file():
            manifest.append({"type": label, "file": p.name, "path": str(p)})

manifest_df = pd.DataFrame(manifest)
manifest_path = MANIFEST_DIR / "manifest_outputs.csv"
manifest_df.to_csv(manifest_path, index=False, encoding="utf-8-sig")

summary = pd.DataFrame({
    "Komponen": ["CSV tables", "PNG tables", "PNG figures", "Audit outputs", "Total files"],
    "Jumlah": [
        len(list(TABLE_DIR.glob("*"))),
        len(list(TABLE_IMG_DIR.glob("*"))),
        len(list(FIG_DIR.glob("*"))),
        len(list(AUDIT_DIR.glob("*"))),
        len(manifest)
    ]
})
summary_path = MANIFEST_DIR / "summary_outputs.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

display(summary)
display(manifest_df)

zip_path = Path("/content/dt_routing_outputs_ayub.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(OUTPUT_DIR)))

print("ZIP akhir berhasil dibuat:", zip_path)

if IN_COLAB:
    files.download(str(zip_path))